# 파이널 프로젝트 : 은행 이탈 고객 예측



- **분류**
- 회귀

- 데이터
    - URL : https://www.kaggle.com/competitions/playground-series-s4e1
    - train : 훈련 데이터 세트
    - test : 테스트 데이터 세트
    - sample_submission : 올바른 형식의 샘플 제출 파일

- 평가 지표
    - ROC Curve


- 변수
    - Features(X)
        - ID : 순번
        - Customer ID: 각 고객의 고유 식별번호
        - Surname: 고객의 성
        - Credit Score: 고객의 신용점수
        - Geography: 고객이 거주하는 국가
        - Gender: 고객의 성별
        - Age: 고객의 나이
        - Tenure: 고객이 은행을 이용한 기간
        - Balance: 고객의 계좌 잔액
        - NumOfProducts: 고객이 이용하는 은행 상품의 수(ex. 예금,적금)
        - HasCrCard: 신용카드 보유 여부
        - IsActiveMember: 활성 회원 여부
        - EstimatedSalary: 고객의 예상 연봉
    - Target(Y)
        - Exited: 고객 이탈 여부







## 3_2 point
- TF-IDF 사용
- SVD 사용
- TF-IDF, SVD를 통한 파생변수
    - Sur_Geo_Gend_Sal : CustomerId, Surname, Geography, Gender, EstimatedSalary


### 3_2_1) TF-IDF
- **텍스트 데이터를 숫자로 변환**하여 머신러닝 모델이 처리할 수 있게 함
- **단어의 중요도를 계산**하여, 중요한 단어를 높은 점수, 흔한 단어는 낮은 점수를 부여
- TF 작동원리
    - 특정 단어가 한 문장에서 얼마나 자주 등장했는지 비율
- IDF 작동원리
    - 특정 단어가 전체 문서에서 얼마나 흔한지 계산
    - 가중치 이용
- TF-IDF 점수
    - TF * IDF = 최종 점수

```
<예시>
vectorizer = TfidfVectorizer(max_features=1000)  # 최대 1000개의 중요한 단어 선택
vectors_train = vectorizer.fit_transform(df_train[col_name])  # TF-IDF 점수 계산
vectors_test = vectorizer.transform(df_test[col_name])  # 테스트 데이터 변환
```



### 3_2_1) SVD

-  **행열을 분해**하여 데이터의 본질적인 구조를 이해하거나 **차원을 줄이는데** 사용하는 강력한 수학적 기법
-  데이터를 압축하거나 중요한 특징만 추출, 사용하는데 유용
- 특히, 데이터의 패턴을 추출하거나 노이지를 제거하는데 탁월


$$A=UΣV^T$$
- A : 원본 행렬 (크기 m x n)

- U : 왼쪽 직교 행렬 (크기 m x n) --> A의 행(row) 정보를 담고 있음

- Σ : 대각 행렬 (크기 m x n) --> A의 중요한 값(특이값, Singular Values)을 담고 있음

- V^T : 오른쪽 직교 행렬 (크기 n x n) --> A의 열(columns) 정보를 담고 있음


- 역할
    - 1. 원본 데이터를 분해
        - 데이터를 U,Σ,V^T로 나눔으로써 **'핵심패턴', '덜 중요한 정보'를 분리**
    - 2. **핵심 패턴만 남긴다**
        - Σ에 담긴 특이값 중 가장 큰 몇개만 선택하면 데이터의 **주요 정보 유지 및 차원 축소**
    - 3. 다시 원본 데이터로 복원
        - 분해된 U,Σ,V^T 를 사용하여 원본 데이터를 복원

```
      | 85   78    92    85 |
      | 72   70    75    72 |
A =   | 90   88    95    90 |
      | 60   65    62    60 |
      | 95   93    97    95 |


- U : 학생의 시험 점수 패턴을 나타냄
- Σ : 점수 데이터의 중요한 특징(특이값)을 나타냄
- V^T : 각 과목 간의 관계를 나타냄


```

- TruncatedSVD
    - **TF-IDF로 변환된 데이터는 차원이 매우 커짐**
    - **TruncatedSVD로 차원 축소** -> 데이터의 주요 정보 압축
    - 작동원리
        - TF-IDF로 변환된 데이터는 고차원의 희소 행렬(sparse matrix)
            - 희소 행렬 : 대부분의 값이 0인 행렬
        - SVD를 사용 -> 차원 축소
            - 데이터를 **주성분(가장 중요한 정보)으로만 압축 표현**

- 차원 축소하면서 데이터의 핵심 정보 유지
- 모델이 효율적으로 학습 도움

```
<예시>
svd = TruncatedSVD(n_components=3)  # 3개의 주요 정보 축으로 압축
x_sv_train = svd.fit_transform(vectors_train)  # 훈련 데이터를 SVD로 축소
x_sv_test = svd.transform(vectors_test)  # 테스트 데이터 변환
```


### 3_2_1) TF-IDF, SVD를 통한 파생변수-IDF


- Surname : 고객의 성
    - 분석에 있어서 의미없을 가능성 높음
    - 하지만 은행 서비스에서 고객 성씨가 특정 국가나 지역, 문화적 특징과 관련될 가능성이 있음
    - 특정 성씨가 은행 서비스 이용이나 이탈에 영향을 줄 수 있다는 가설로 포함
    - 단순 성씨 분석하는 것보다, 고객 정보를 종합적으로 분석해 더 풍부한 패턴을 학습하려는 시도.

    - 따라서 Surname 사용해서 변수 생성

- Sur_Geo_Gend_Sal : CustomerId, Surname, Geography, Gender, EstimatedSalary
    - int -> str
    - str 유지


## 1.전처리

### 패키지

#### 먼저 설치 후 재시작해야하는 패키지

In [ ]:
!pip install pycaret
!pip install optuna
!pip install catboost

#### 일반 패키지

In [ ]:
# 기본 패키지
import pandas as pd
import numpy as np

# 설정
pd.set_option('display.max_columns', None)  # 최대 컬럼 설정

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 출력 관련(이미지 표시)
from IPython.display import Image

# 계층적 클러스터링
import scipy.cluster.hierarchy as sch  # 계층적 군집화 알고리즘

# 데이터 전처리
## 인코딩
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder # pd.get_dummies로 OneHotEncoder 대체
## 스케일링
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler

# 데이터 분할
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

# 다중공선성 확인(VIF)
from statsmodels.stats.outliers_influence import variance_inflation_factor

# PyCaret: AutoML (자동화된 머신러닝 워크플로우, 모델 선택)
from pycaret.classification import *

# 텍스트 데이터 처리
from sklearn.feature_extraction.text import TfidfVectorizer  # 텍스트 데이터 TF-IDF 벡터화
from sklearn.decomposition import TruncatedSVD  # 차원 축소

# Optuna: 하이퍼파라미터 최적화
import optuna
from optuna.samplers import TPESampler  # 샘플링 알고리즘 (TPE)

# 머신러닝 모델
## GBR
from sklearn.ensemble import GradientBoostingClassifier
## AdaBoost
from sklearn.ensemble import AdaBoostClassifier
## LightGBM
from lightgbm import LGBMClassifier
import lightgbm as lgb
## DecisionTree
from sklearn.tree import DecisionTreeClassifier
## XGBoost
from xgboost import XGBClassifier
import xgboost as xgb
## CatBoost
from catboost import CatBoostClassifier, Pool # pool : catboost에서 데이터 관리위한 특수 클래스(데이터 컨테이너), 범주 변수 자동 인식, 데이터 빠른 처리 등
from catboost.utils import eval_metric

# 성능 평가 지표
from sklearn.metrics import roc_auc_score  # ROC AUC 점수 계산

# 모델 앙상블
from sklearn.ensemble import VotingClassifier  # 앙상블 학습을 위한 VotingClassifier

# 조합 생성 함수
from itertools import combinations

# 변수 중요도
from sklearn.inspection import permutation_importance


### 데이터 불러오기
- train
- test
- submission

In [ ]:
train = pd.read_csv('/content/drive/MyDrive/파이널 프로젝트_이정수/분류/train.csv')
test = pd.read_csv('/content/drive/MyDrive/파이널 프로젝트_이정수/분류/test.csv')
submission = pd.read_csv('/content/drive/MyDrive/파이널 프로젝트_이정수/분류/sample_submission.csv')

#### 데이터 파악

##### 데이터 크기

- train
    - 데이터 : 165034
    - 변수 : 14
- test
    - 데이터 : 110023
    - 변수 : 13
- submission
    - 데이터 : 110023
    - 변수 : 2

In [ ]:
for i in [train, test,submission] :
  print(i.shape)

##### 데이터 타입, 결측

- **train**
  - 결측치
      - 0개
  - 타입
      - Int64, float64 : 11개
      - object : 3개
          - Surname : 고객의 성
          - Geography : 고객이 거주하는 국가
          - Gender : 고객의 성별
- **test**
  - 결측치
      - 0개
  - 타입
      - train과 동일(Exited 제외)
- **submission**
  - 결측치
      - 0개
  - 타입
      - int64, float64

### 시각화
- **Barplot**
    - Surname : 범주형 변수이지만 고유값 수가 2797개로 부적합
    - Geography
    - Gender
    - NumOfProducts
    - HasCrCard
    - IsActiveMember
    - Exited
- **Histplot, 밀도추정그래프, BoxPlot, ScatterPlot**
    - CreditScore
    - Age
    - Tenure
    - Balance
    - EstimatedSalary

- **상관관계 히트맵**

#### 시각화 함수 생성

In [ ]:
# 1. BarPlot
def custom_barplot(ax, count):
    for bar in ax.patches:                                                            # ax.patches : 그래프 그려지는 모든 도형 요소
        if bar.get_height() == 0: continue                                            # 막대의 높이가 0이면 건너뜀
        rate = bar.get_height() / count * 100                                         # 백분율 계산

        ax.text(x=bar.get_x() + bar.get_width() / 2,                                  # 막대의 중앙 위치 : bar.get_x(막대의 왼쪽 x좌표), bar.get_width(막대의 너비 = 가로 길이)
                y=bar.get_height() + count * 0.005,                                   # 막대 위쪽에 텍스트 위치 : bar.get_height(막대의 높이 = 세로 길이)
                s=f'{rate:1.1f}%', ha='center')                                       # 비율 텍스트 추가

    ax.set_ylabel('')                                                                 # y축 레이블 제거
    return ax                                                                         # 수정된 Axes 객체 반환

# 2. Histogram
def custom_histogram(ax, data):
    mean = data.mean()                                                                # 데이터 평균 계산
    ax.text(x=ax.get_xlim()[0] + 0.05 * (ax.get_xlim()[1] - ax.get_xlim()[0]),        # 왼쪽 여백
            y=max([bar.get_height() for bar in ax.patches]) * 0.85,                   # y 위치 조정
            s=f'Mean: {mean:.2f}', ha='left', color='red')                            # 평균값 표시

    ax.set_ylabel('')                                                                 # y축 레이블 제거
    return ax                                                                         # 수정된 Axes 객체 반환

# 3. Box Plot
def custom_boxplot(ax, data):
    y_median = np.median(data)                                                        # 중앙값 계산
    ax.text(x=1.02, y=y_median, s=f'Median: {y_median:.2f}', ha='left', color='red')  # 중앙값 표시


##### train 시각화


In [ ]:
fig, axes = plt.subplots(6, 2, figsize=(15, 20))

# Credit Score Distribution
sns.histplot(train['CreditScore'], bins=30, kde=True, ax=axes[0, 0])
custom_histogram(axes[0, 0], train['CreditScore'])  # CreditScore 데이터 전달
axes[0, 0].set_title('Credit Score Distribution')

# Geography Distribution
ax_geo = sns.countplot(data=train, x='Geography', hue='Exited', ax=axes[0, 1])
custom_barplot(ax_geo, train.shape[0])  # 비율 추가
axes[0, 1].set_title('Geography Distribution')

# Gender Distribution
ax_gender = sns.countplot(data=train, x='Gender', hue='Exited', ax=axes[1, 0])
custom_barplot(ax_gender, train.shape[0])  # 비율 추가
axes[1, 0].set_title('Gender Distribution')

# Age Distribution
sns.histplot(train['Age'], bins=30, kde=True, ax=axes[1, 1])
custom_histogram(axes[1, 1], train['Age'])  # Age 데이터 전달
axes[1, 1].set_title('Age Distribution')

# Tenure Distribution
ax_tenure = sns.countplot(data=train, x='Tenure', hue='Exited', ax=axes[2, 0])
custom_barplot(ax_tenure, train.shape[0])  # 비율 추가
axes[2, 0].set_title('Tenure Distribution')

# Balance Distribution
sns.histplot(train['Balance'], bins=30, kde=True, ax=axes[2, 1])
custom_histogram(axes[2, 1], train['Balance'])  # Balance 데이터 전달
axes[2, 1].set_title('Balance Distribution')

# NumOfProducts Distribution
ax_num_products = sns.countplot(data=train, x='NumOfProducts', hue='Exited', ax=axes[3, 0])
custom_barplot(ax_num_products, train.shape[0])  # 비율 추가
axes[3, 0].set_title('NumOfProducts Distribution')

# HasCrCard Distribution
ax_has_crcard = sns.countplot(data=train, x='HasCrCard', hue='Exited', ax=axes[3, 1])
custom_barplot(ax_has_crcard, train.shape[0])  # 비율 추가
axes[3, 1].set_title('HasCrCard Distribution')

# IsActiveMember Distribution
ax_active_member = sns.countplot(data=train, x='IsActiveMember', hue='Exited', ax=axes[4, 0])
custom_barplot(ax_active_member, train.shape[0])  # 비율 추가
axes[4, 0].set_title('IsActiveMember Distribution')

# Estimated Salary Distribution
sns.histplot(train['EstimatedSalary'], bins=30, kde=True, ax=axes[4, 1])
custom_histogram(axes[4, 1], train['EstimatedSalary'])  # Estimated Salary 데이터 전달
axes[4, 1].set_title('Estimated Salary Distribution')

# Exited Distribution
ax_exited = sns.countplot(data=train, x='Exited', ax=axes[5, 0])
custom_barplot(ax_exited, train.shape[0])  # 비율 추가
axes[5, 0].set_title('Exited Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# 박스플롯을 그릴 수직 축 배열 설정
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

# Credit Score 박스플롯
sns.boxplot(y=train['CreditScore'], ax=axes[0, 0])
custom_boxplot(axes[0, 0], train['CreditScore'])  # 중앙값 텍스트 추가
axes[0, 0].set_title('Credit Score')

# Age 박스플롯
sns.boxplot(y=train['Age'], ax=axes[0, 1])
custom_boxplot(axes[0, 1], train['Age'])  # 중앙값 텍스트 추가
axes[0, 1].set_title('Age')

# Balance 박스플롯
sns.boxplot(y=train['Balance'], ax=axes[1, 0])
custom_boxplot(axes[1, 0], train['Balance'])  # 중앙값 텍스트 추가
axes[1, 0].set_title('Balance')

# Estimated Salary 박스플롯
sns.boxplot(y=train['EstimatedSalary'], ax=axes[1, 1])
custom_boxplot(axes[1, 1], train['EstimatedSalary'])  # 중앙값 텍스트 추가
axes[1, 1].set_title('Estimated Salary')

plt.tight_layout()  # 그래프 사이의 간격을 조정하여 보기 좋게 배치
plt.show()

###### train 시각화 해석

- BarPlot, HistPlot 해석
    - **Credit Score(신용점수)**
        - 평균
            - 656.45
        - 분포
            - 대체로 정규분포 형태
            - 신용 점수 분포가 넓게 퍼져있어 다양한 점수 분포
            - 신용점수가 600-700 사이에 가장 많은 고객이 분포

    - **Geography(고객 거주 국가)**
        - 분포
            - France가 전체의 57.1%로 가장 많고, 그 다음 Spain 22%, Germany 20.9% 순서
        - 이탈
            - 세 국가 모두 이탈하지 않은 고객(0)이 이탈 고객(1)보다 많다
                - 프랑스는 이탈 비율이 낮고, 독일은 상대적으로 높은 이탈율을 보인다

    - **Gender(고객 성별)**
        - 분포
            - 남성이 약 56.5%, 여성이 43.6%로 대략 6:4 비율
        - 이탈
            - 남성, 여성 모두 이탈하지 않은 사람(0)이 이탈자(1)보다 많다
                - 남성보다 여성이 이탈 비율이 더 높다

    - **Age(고객 연령)**
        - 평균
            - 38.13
        - 분포
            - 대체로 정규분포 형태
            - 30,40대에 주 고객층이 모여있다
            - 나이가 많을 수록 고객 수가 급격히 감소하는 경향
                - 젊은 고객층(30-40대)에서 고객수가 집중

    - **Tenure(은행 이용 기간)**
        - 분포
            - 모든 기간에 고객수가 고르게 분포되어 있지만, 0년과 10년에 소폭 감소
        - 이탈
            - 이용 기간에 관계없이 이탈하지 않은 사람이(0)이 이탈자(1) 보다 많다

    - **Balance(계좌잔액)**
        - 평균
            - 55478.09
        - 분포
            - 0인 경우가 가장 많으며, 0을 제외한 나머지 값들은 정규분포에 가깝게 분포
            - 주로 50,000~200,000 구간에 집중

    - **NumOfProducts(고객 이용 은행 상품 수)**
        - 분포
            - 1개 또는 2개 상품을 보유한 고객이 대부분
                - 1개(46.9%), 2개(51.1%)
        - 이탈
            - 상품 개수가 1개인 고객의 이탈 비율이 상대적으로 높다

    - **HasCrCard(신용카드 보유 여부)**
        - 분포
            - 신용카드 보유 고객(1)이 75.4%, 비보유 고객(0)이 24.6%의 분포로 보유한 고객이 더 많다
        - 이탈
            - 신용카드 보유 여부와 관계없이 이탈하지 않은 고객이 더 많다.

    - **IsActiveMemebr(활성 회원 여부)**
        - 분포
          - 비활성회원이 50.2%, 활성회원이 49.7%로 차이가 매우 적었다.
        - 이탈
            - 비활성 회원의 이탈율이 더 높다.

    - **EstimatedSalary(추정 연봉)**
        - 평균
            - 112574.82
        - 분포
            - 급여가 0에서 200000까지 고르게 분포
                - 주요 구간 : 50000 - 175000

    - **Exited(이탈 여부)**
        - 분포
            - 전체 고객 중 이탈하지 않은 사람(0)이 78.8%, 이탈자(1)는 21.2%로 대략 8:2 비율

- BoxPlot 해석
    - CreditScore
        - 이상치
            - O
            - 신용 점수 하단에 이상치 다수 존재
                - 특히 400이하 낮은 점수 고객, 점수가 낮은 일부 고객층 포함
        - 중앙값
            - 659
        - 분포
            - 약간 오른쪽으로 치우친 분포
                - 왼쪽꼬리

    - Age
        - 이상치
            - O
            - 60 이상의 나이에서 이상치 다수 존재
                - 특히 80세 이상 고객이 드물게 포함, 고령층 소수 포함
        - 중앙값
            - 37
        - 분포
            - 오른쪽 긴 꼬리
    - Balance
        - 이상치
            - X
        - 중앙값
            - 0
    - EstimatedSalary
        - 이상치
            - X
        - 중앙값
            - 117,948

##### test 시각화


In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(15, 20))

# Credit Score Distribution
sns.histplot(test['CreditScore'], bins=30, kde=True, ax=axes[0, 0])
custom_histogram(axes[0, 0], test['CreditScore'])  # CreditScore 데이터 전달
axes[0, 0].set_title('Credit Score Distribution')

# Geography Distribution (without hue)
ax_geo = sns.countplot(data=test, x='Geography', ax=axes[0, 1])
custom_barplot(ax_geo, test.shape[0])  # 비율 추가
axes[0, 1].set_title('Geography Distribution')

# Gender Distribution (without hue)
ax_gender = sns.countplot(data=test, x='Gender', ax=axes[1, 0])
custom_barplot(ax_gender, test.shape[0])  # 비율 추가
axes[1, 0].set_title('Gender Distribution')

# Age Distribution
sns.histplot(test['Age'], bins=30, kde=True, ax=axes[1, 1])
custom_histogram(axes[1, 1], test['Age'])  # Age 데이터 전달
axes[1, 1].set_title('Age Distribution')

# Tenure Distribution (without hue)
ax_tenure = sns.countplot(data=test, x='Tenure', ax=axes[2, 0])
custom_barplot(ax_tenure, test.shape[0])  # 비율 추가
axes[2, 0].set_title('Tenure Distribution')

# Balance Distribution
sns.histplot(test['Balance'], bins=30, kde=True, ax=axes[2, 1])
custom_histogram(axes[2, 1], test['Balance'])  # Balance 데이터 전달
axes[2, 1].set_title('Balance Distribution')

# NumOfProducts Distribution (without hue)
ax_num_products = sns.countplot(data=test, x='NumOfProducts', ax=axes[3, 0])
custom_barplot(ax_num_products, test.shape[0])  # 비율 추가
axes[3, 0].set_title('NumOfProducts Distribution')

# HasCrCard Distribution (without hue)
ax_has_crcard = sns.countplot(data=test, x='HasCrCard', ax=axes[3, 1])
custom_barplot(ax_has_crcard, test.shape[0])  # 비율 추가
axes[3, 1].set_title('HasCrCard Distribution')

# IsActiveMember Distribution (without hue)
ax_active_member = sns.countplot(data=test, x='IsActiveMember', ax=axes[4, 0])
custom_barplot(ax_active_member, test.shape[0])  # 비율 추가
axes[4, 0].set_title('IsActiveMember Distribution')

# Estimated Salary Distribution
sns.histplot(test['EstimatedSalary'], bins=30, kde=True, ax=axes[4, 1])
custom_histogram(axes[4, 1], test['EstimatedSalary'])  # Estimated Salary 데이터 전달
axes[4, 1].set_title('Estimated Salary Distribution')

plt.tight_layout()
plt.show()


In [ ]:
# 박스플롯을 그릴 수직 축 배열 설정
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

# Credit Score 박스플롯
sns.boxplot(y=test['CreditScore'], ax=axes[0, 0])
custom_boxplot(axes[0, 0], test['CreditScore'])  # 중앙값 텍스트 추가
axes[0, 0].set_title('Credit Score')

# Age 박스플롯
sns.boxplot(y=test['Age'], ax=axes[0, 1])
custom_boxplot(axes[0, 1], test['Age'])  # 중앙값 텍스트 추가
axes[0, 1].set_title('Age')

# Balance 박스플롯
sns.boxplot(y=test['Balance'], ax=axes[1, 0])
custom_boxplot(axes[1, 0], test['Balance'])  # 중앙값 텍스트 추가
axes[1, 0].set_title('Balance')

# Estimated Salary 박스플롯
sns.boxplot(y=test['EstimatedSalary'], ax=axes[1, 1])
custom_boxplot(axes[1, 1], test['EstimatedSalary'])  # 중앙값 텍스트 추가
axes[1, 1].set_title('Estimated Salary')

plt.tight_layout()  # 그래프 사이의 간격을 조정하여 보기 좋게 배치
plt.show()

###### train 시각화 해석

- BarPlot, HistPlot 해석
    - - **Credit Score(신용점수)**
    - 평균
        - 656.53
    - 분포
        - 대체로 정규분포 형태
        - 신용 점수 분포가 넓게 퍼져있어 다양한 점수 분포
        - 신용점수가 600-700 사이에 가장 많은 고객이 분포

- **Geography(고객 거주 국가)**
    - 분포
        - France가 전체의 57.4%로 가장 많고, 그 다음 Spain 21.7%, Germany 20.8% 순서

- **Gender(고객 성별)**
    - 분포
        - 남성이 약 56.3%, 여성이 43.7%

- **Age(고객 연령)**
    - 평균
        - 38.12
    - 분포
        - 대체로 정규분포 형태
        - 30,40대에 주 고객층이 모여있다
        - 연령이 증가할수록 고객 수가 급격히 감소하는 경향
            - 젊은 고객층(30-40대)에서 고객수가 집중

- **Tenure(은행 이용 기간)**
    - 분포
        - 모든 기간에 고객수가 고르게 분포되어 있지만, 0년과 10년에 소폭 감소

- **Balance(계좌잔액)**
    - 평균
        - 55333.61
    - 분포
        - 0인 경우가 가장 많으며, 0을 제외한 나머지 값들은 정규분포에 가깝게 분포
        - 주로 50,000~200,000 구간에 집중

- **NumOfProducts(고객 이용 은행 상품 수)**
    - 분포
        - 1개 또는 2개 상품을 보유한 고객이 대부분
            - 1개(46.9%), 2개(51.2%)

- **HasCrCard(신용카드 보유 여부)**
    - 분포
        - 신용카드 보유 고객(1)이 73.3%, 비보유 고객(0)이 24.7%의 분포로 보유한 고객이 더 많다

- **IsActiveMemebr(활성 회원 여부)**
    - 분포
      - 비활성회원이 50.5%, 활성회원이 49.5%로 차이가 매우 적었다.

- **EstimatedSalary(추정 연봉)**
    - 평균
        - 112315.15
    - 분포
        - 급여가 0에서 200000까지 고르게 분포
            - 주요 구간 : 50000 - 175000
- BoxPlot 해석
    - CreditScore
        - 이상치
            - O
            - 신용 점수 하단에 이상치 다수 존재
                - 특히 400이하 낮은 점수 고객, 점수가 낮은 일부 고객층 포함
        - 중앙값
            - 660
        - 분포
            - 약간 오른쪽으로 치우친 분포
                - 왼쪽꼬리

    - Age
        - 이상치
            - O
            - 60 이상의 나이에서 이상치 다수 존재
                - 특히 80세 이상 고객이 드물게 포함, 고령층 소수 포함
        - 중앙값
            - 37
        - 분포
            - 오른쪽 긴 꼬리
    - Balance
        - 이상치
            - X
        - 중앙값
            - 0
    - EstimatedSalary
        - 이상치
            - X
        - 중앙값
            - 117,832.23

### 로그변환 & 스케일링
```
- StandardScaler : Mean = 0, std = 1
    - 데이터 정규분포에 가까운 경우
    - 변수 크기 차이가 크고, 이상치 적을 때

- RobustScaler : Median, IQR 사용
    - 이상치 영향 최소화
    - 중앙값 기준으로 스케일링 하므로 분포가 비대칭적인 경우 효율
    - 데이터가 정규분포를 따르는 경우, StandarScaler만큼 효과적이지 않을 수 있다

- MinMaxScaler : Min = 0 , Max = 1 사이 스케일링
    - 데이터가 특정 범위내로 제한되어야 할 때
    - 값의 분포가 균등하지 않을때
    - 딥러닝 모델에서 주로 사용
    - 해석이 쉽다
    - 값의 분포를 유지하며 스케일링
    - 이상치에 민감
```

- Balance, Age : 로그변환
- CreditScore, Age, Balance, EstimatedSalary : 스케일링
    - CreditScore, Age
        - RobustScaler
    - Balance
        - MinMaxScaler
    - estimatedSalary
        - StandardScaler


In [ ]:
numeric_cols = ['Age','CreditScore','Balance','EstimatedSalary']

# 로그 변환과 스케일링을 포함한 기능 정의
scaled_features = {
    'CreditScore': RobustScaler(),
    'EstimatedSalary': StandardScaler(),
    'Age': ('log', RobustScaler()),        # 로그 변환 후 스케일링
    'Balance': ('log1p', MinMaxScaler())   # 로그 변환 후 스케일링
}

# 스케일링 적용
for feature, scaled_info in scaled_features.items() :
  if isinstance(scaled_info, tuple) :
    log_info, scaler = scaled_info
    if log_info == 'log' :
      train_feature = np.log(train[[feature]])
      test_feature = np.log(test[[feature]])
    elif log_info == 'log1p' :
      train_feature = np.log1p(train[[feature]])
      test_feature = np.log1p(test[[feature]])
  else :
    scaler = scaled_info
    train_feature = train[[feature]]
    test_feature = test[[feature]]


  # 스케일링 적용
  scaled_feature = feature + '_Scaled'
  train[scaled_feature] = scaler.fit_transform(train_feature)
  test[scaled_feature] = scaler.transform(test_feature)

### 파생변수
- kaagle 참고
    - https://www.kaggle.com/code/abdmental01/bank-churn-lightgbm-and-catboost-0-8945
- tf-idf
- svd

#### TF-IDF & SVD용 변수조합
- CustomerID + Surname + Geography + Gender + EstimtaedSalary

In [ ]:
# 변수 합치기
train['Sur_Geo_Gend_Sal'] = train['CustomerId'].astype('str') + train['Surname'] + train['Geography'] + train['Gender'] + np.round(train.EstimatedSalary).astype('str')
test['Sur_Geo_Gend_Sal'] = test['CustomerId'].astype('str') + test['Surname'] + test['Geography'] + test['Gender'] + np.round(test.EstimatedSalary).astype('str')

#### TF-IDF & SVD

In [ ]:
# 특정 텍스트 열(col_name)을 TF-IDF로 벡터화하고, 차원축소를 통해 주요 특징만 추출하는 과정
def get_vectors(train,test,col_name):

    # TF-IDF 벡터화
    '''
    TF-IDF 코드 분해
    : 상위 1000개 주요 단어만 벡터화 사용
    : 훈련데이터에 대해 벡터화 학습 및 변환 수행(벡터화 -> 텍스트를 중요도 점수로 표시) ->  # TF-IDF 점수 계산
    : 테스트 데이터 훈련데이터 기준으로 변환 수행
    '''
    vectorizer = TfidfVectorizer(max_features=1000)
    vectors_train = vectorizer.fit_transform(train[col_name])
    vectors_test = vectorizer.transform(test[col_name])

    # TruncatedSVD 차원축소
    '''
    TruncatedSVD 코드 설명 및 분해
    <TruncatedSVD 설명>
    : 차원 축소, U⋅Σ⋅V^T -> 이 중 중요한 k개의 차원만 남긴다
    : 학습 -> 데이터의 주요 패턴(특이값) 학습 -> 학습된 U,Σ,𝑉^T를 사용해 주요 정보만 남기기
    : 변환 -> 학습 정보 바탕으로 원래 vectors_train을 저차원 벡터 공간으로 변환

    ```
    원래 TF-IDF 행렬 : 책을 단어들의 숫자로 표현
    - 행 : 문서 /  열 : 단어 /  값 : TF-IDF 점수
    - 크기 (1000,5000)

    SVD 학습 및 변환 후 : "이 책은 과학, 우주 기술"과 같이 책의 핵심 주제를 3개의 숫자로 요약
    - 행 : 문서 / 열 : 중요 특성(3차원)
    - 크기 (1000,3)
    ```
    <TruncatedSVD 코드 분해>
    : TF-IDF 행렬을 3개의 중요 성분으로 압축 -> 데이터 크기 줄이기, 모델 학습속도 개선, 과적합방지
    : vectors_train에 대해 SVD 학습(fit) 및 변환
    '''
    svd = TruncatedSVD(3)
    x_sv_train = svd.fit_transform(vectors_train)
    x_sv_test = svd.transform(vectors_test) # 테스트 데이터 반환


    # 데이터 프레임 변환
    tfidf_df_train = pd.DataFrame(x_sv_train)
    tfidf_df_test = pd.DataFrame(x_sv_test)


    # 열이름 지정
    cols = [(col_name + "_tfidf_" + str(f)) for f in tfidf_df_train.columns.to_list()]
    tfidf_df_train.columns = cols
    tfidf_df_test.columns = cols

    # Reset the index of the DataFrames before concatenation
    train = train.reset_index(drop=True)
    test = test.reset_index(drop=True)

    # Concatenate transformed features with original data
    train = pd.concat([train, tfidf_df_train], axis="columns")
    test = pd.concat([test, tfidf_df_test], axis="columns")
    return train,test

In [ ]:
train,test = get_vectors(train,test,'Surname')
train,test = get_vectors(train,test,'Sur_Geo_Gend_Sal')

#### 변수 조합 및 인코딩
- Senior
    - Age >= 60 : 1
    - Age < 60 : 0
- Active_by_CreditCard
    - HasCrCard * IsActiveMember
- Products_Per_Tenure
    - Tenure * NumOfProducts
- AgeCat
    - 연령 20살 단위로 분해
        - 0-9 : 0
        - 10-29 : 1
        - 30-49 : 2
        - 50-69 : 3
        - 70-89 : 4
        - 90-109 : 5

In [ ]:
# 파생변수 생성 및 인코딩
def feature_data(df):
    # senior 생성(60대 이상은 1, 아니면 0)
    df['Senior'] = df['Age'].apply(lambda x: 1 if x >= 60 else 0)
    # 신용카드 소유자와 활성화 고객 여부
    df['Active_by_CreditCard'] = df['HasCrCard'] * df['IsActiveMember']
    # 상품수 대비 이용기간
    df['Products_Per_Tenure'] =  df['Tenure'] / df['NumOfProducts']
    # 새로운 연령대 범주(20살 단위)
    df['AgeCat'] = np.round(df.Age/20).astype('int').astype('category')

    cat_cols = ['Geography', 'Gender', 'NumOfProducts','AgeCat']

    #onehotEncoding
    df=pd.get_dummies(df,columns=cat_cols)
    return df

In [ ]:
train = feature_data(train)
test = feature_data(test)

feat_cols = train.columns.drop(['id','CustomerId','Surname','Exited','Sur_Geo_Gend_Sal'])
feat_cols = feat_cols.drop(numeric_cols)

feat_cols_test = test.columns.drop(['id','CustomerId','Surname','Sur_Geo_Gend_Sal'])
feat_cols_test = feat_cols_test.drop(numeric_cols)

In [ ]:
train_feature = train[feat_cols]
train_target = train['Exited']
test = test[feat_cols_test]

In [ ]:
for i in train_feature.columns :
  if train_feature[i].dtype == 'bool' :
    train_feature[i] = train_feature[i].astype('int')

for i in test.columns :
  if test[i].dtype == 'bool' :
    test[i] = test[i].astype('int')

### 다중공선성(VIF)
- 5 - 10
    - Tenure : 9.336355
    - Active_by_CreditCard : 5.035984
- vif >10
    - Products_Per_Tenure : 12.44187
    
- 의미적으로 유의할 수 있는 가능성이 있으므로 유지하고 분석 진행
    - 최종 모델에서의 feature_importance, permutation_importance를 통해 제거 여부 재차 확인

In [ ]:
# 필요한 패키지 import(분산팽창계수(VIF))
from statsmodels.stats.outliers_influence import variance_inflation_factor

# VIF 계산 : (exog : Any, exog_idx : Any ) -> (독립변수, 독립변수 인덱스) -> 독립변수 : 독립 변수들을 포함하는 2D 배열이나 데이터프레임 /
vif_data = pd.DataFrame()
vif_data['Feature'] = train_feature.columns
vif_data['VIF'] = [variance_inflation_factor(train_feature.values, i) for i in range(train_feature.shape[1])]
print(vif_data)

### 추가 시각화_상관관계 히트맵
- **Exited**
    - **양**
        - **Age_Scaled(0.34)**
            - 연령이 높을수록 이탈 가능성이 다소 증가
        - NumOfProducts_1(0.31)
            - 이용중인 상품이 1개일 경우 이탈 가능성 다소 증가
        - AgeCat_2(0.26)
            - 30세에서 49세 사이의 고객의 이탈 가능성 다소 증가
    - **음**
        - **NumOfProducts_2(-0.38)**
            - 이용중인 상품이 2개일 경우 이탈 가능성이 다소 감소
        - IsActiveMember(-0.21)
            - 활성 멤버일 수록 이탈 가능성이 다소 감소
        - Active_By_CreditCard(-0.18)
            - 신용카드를 보유한 고객 중 활성 고객은 이탈 가능성이 다소 낮음
- **전체 상관계수**
    - 다수의 파생변수 생성으로 변수간 상관관계가 복잡해지고 중복성이 높아짐
    - 수치의 신뢰성과 해석력 저하
    - 전체 상관계수 분석은 참고용으로 활용

In [ ]:
# 임시로 train_feature, train_target 합치기
temp = pd.concat([train_feature, train_target], axis=1)
corr = temp.corr()

# 히트맵 시각화
plt.figure(figsize=(30, 27))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', center=0, cbar=True)

# 그래프 제목 추가
plt.title('Correlation Heatmap')

# 그래프 출력
plt.show()

## 2.모델링

### 데이터 분리

In [ ]:
# 필요한 패키지 import
from sklearn.model_selection import train_test_split

# 데이터 분리
X_train, X_valid, y_train, y_valid = train_test_split(train_feature, train_target,
                                                      test_size = 0.3,
                                                      random_state=0)

# 분리 데이터 크기 확인
for i in [X_train, X_valid, y_train, y_valid] :
  print(i.shape)

### AutoML
- catboost : CatBoost Classifier
- lighbgbm : Light Gradient Boosting Machine
- gbc : Gradient Boosting Classifier
- xgboost : Extreme Gradient Boosting
- ada : Ada Boost Classifier

In [ ]:
# 인덱스 번호 새로 할당
X_train_automl = X_train.reset_index(drop=True)
y_train_automl = y_train.reset_index(drop=True)

# X_train, y_train 합치기
Xy_train_automl = pd.concat([X_train_automl, y_train_automl], axis=1)

# AutoML 모델 세팅
clf = setup(data=Xy_train_automl,
            target = 'Exited',
            train_size = 0.7,
            data_split_shuffle=True,
            session_id = 0,
            fold = 5
            )

# AutoML top5 model 설정
top5_model = compare_models(fold=5, round=3, n_select=5, sort = 'AUC', errors='ignore',verbose=True)
top5_model

In [ ]:
from IPython.display import Image
Image('/content/drive/MyDrive/파이널 프로젝트_이정수/분류/SVD_AutoML.png',width=800)

### Optuna
- Catboost
```
Best AUC: 0.8929926718960395
Best hyperparameters:
  iterations: 567
  learning_rate: 0.02900858676615259
  depth: 7
  l2_leaf_reg: 1.2647515608981543
  random_strength: 0.23498389010132387
  bagging_temperature: 0.024432250592590334
  grow_policy: Depthwise
  border_count: 174
  od_wait: 50
```
- gbc
```
Best AUC: 0.8872064342320748
Best hyperparameters:
  n_estimators: 473
  learning_rate: 0.021945351322294467
  max_depth: 10
  min_samples_split: 2
  min_samples_leaf: 3
  subsample: 0.9750612643600708
  max_features: sqrt
  loss: exponential
  ccp_alpha: 0.00015702247402449507
  validation_fraction: 0.2984753709846782
  n_iter_no_change: 15
  tol: 0.005547267099418254
  min_impurity_decrease: 0.09411712873217182
  max_leaf_nodes: 86
```
- lightgbm
```
Best AUC: 0.8932907005145936
Best hyperparameters:
  num_boost_round: 633
  learning_rate: 0.03342462805497592
  num_leaves: 56
  max_depth: 14
  min_data_in_leaf: 52
  feature_fraction: 0.6364882192462488
  bagging_fraction: 0.8584003437311537
  bagging_freq: 2
  min_gain_to_split: 0.14446805858185238
  lambda_l1: 0.07945719422821337
  lambda_l2: 0.09228860158082569
  tree_learner: feature
  max_bin: 277
  early_stopping_rounds: 15
  num_threads: 8
  scale_pos_weight: 4.955235602871113
```
- xgboost
```
Best AUC: 0.8862153263809549
Best hyperparameters:
  num_round: 360
  alpha: 0.7558611471242505
  base_score: 0.5566055700158926
  booster: gbtree
  colsample_bylevel: 0.652980718854562
  colsample_bynode: 0.6207456906259616
  colsample_bytree: 0.7084669537044522
  eta: 0.2931701770417505
  eval_metric: auc
  gamma: 0.38038567264515777
  grow_policy: lossguide
  lambda: 7.6102845156108465
  max_bin: 142
  max_delta_step: 7
  max_depth: 11
  max_leaves: 33
  min_child_weight: 3.6279534873415384
  objective: binary:logistic
  scale_pos_weight: 4.199769049496863
  seed: 675
  subsample: 0.7974634701882836
  verbosity: 1
  early_stopping_rounds: 100
```

- ada
```
Best AUC: 0.8885549835886202
Best hyperparameters:
  n_estimators: 339
  learning_rate: 0.028444996574740207
  algorithm: SAMME.R
  random_state: 582
  max_depth: 5
  min_samples_split: 8
  min_samples_leaf: 3
  max_features: sqrt
  max_leaf_nodes: 12
  min_impurity_decrease: 0.00011591970091207504
```


#### CatBoost : CatBoost Classifier

In [ ]:
sampler = TPESampler(seed=0)

def objective(trial):
    # 하이퍼파라미터 정의
    params = {
        "iterations": trial.suggest_int("iterations", 100, 1000),                                               # 트리 개수 (Boosting 반복 횟수) / default: 1000
        "learning_rate": trial.suggest_loguniform("learning_rate", 1e-4, 1.0),                                  # 학습률 (트리의 가중치를 조정하는 비율) / default: 0.03
        "depth": trial.suggest_int("depth", 1, 16),                                                             # 트리의 최대 깊이 (과적합 방지) / default: 6
        "l2_leaf_reg": trial.suggest_loguniform("l2_leaf_reg", 1e-2, 10.0),                                     # L2 정규화 계수 (과적합 방지) / default: 3.0
        "random_strength": trial.suggest_loguniform("random_strength", 1e-2, 10.0),                             # 트리 분할 시 무작위성 정도 (과적합 방지) / default: 1.0
        "bagging_temperature": trial.suggest_loguniform("bagging_temperature", 1e-5, 10.0),                     # 베이지안 부트스트랩 샘플링 강도 (1 이상이면 샘플링 강화) / default: 1.0
        "grow_policy": trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]),   # 트리 성장 정책 (균형 성장 vs 깊이 우선 vs 손실 기반) / default: SymmetricTree
        "border_count": trial.suggest_int("border_count", 1, 255),                                              # 연속형 피처를 구간화할 때 사용할 분할 개수 / default: 254
        "thread_count": -1,                                                                                     # 사용할 CPU 스레드 개수 (-1이면 모든 코어 사용) / default: -1
        "random_seed": 42,                                                                                      # 재현 가능성을 위한 난수 시드 고정 / default: 없음
        "eval_metric": "AUC",                                                                                   # 평가 지표 (Area Under Curve) / default: Logloss
        "verbose": 0,                                                                                           # 학습 과정 출력 여부 (0이면 출력 안 함)
        "od_type": "Iter",                                                                                      # 조기 종료 조건 (Iteration 기반) / default: IncToDec
        "od_wait": trial.suggest_int("od_wait", 10, 50),                                                        # 조기 종료를 위한 대기 스텝 수 / default: 50
        "task_type": "CPU",                                                                                     # 실행 환경 설정 (CPU 사용)
        "loss_function": "Logloss"                                                                              # 손실 함수 (이진 분류 문제에서는 Logloss 사용)
    }


    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    auc_list = []

    for _, (train_index, valid_index) in enumerate(cv.split(train_feature,train_target)):
        X_train, y_train = train_feature.iloc[train_index], train_target.iloc[train_index]
        X_valid, y_valid = train_feature.iloc[valid_index], train_target.iloc[valid_index]


        cat_model = CatBoostClassifier(**params)
        cat_model.fit(X_train, y_train,
                      eval_set = (X_valid, y_valid),early_stopping_rounds = 50, verbose = 0)

        # 예측 확률 계산
        y_score = cat_model.predict_proba(X_valid)[:,1]
        auc_score = roc_auc_score(y_valid, y_score)
        auc_list.append(auc_score)

    # 평균 auc 반환
    mean_auc = np.mean(auc_list)
    return mean_auc

# optuna 최적화 실행(AUC 최대화)
optuna_cat = optuna.create_study(direction='maximize', sampler=sampler)
optuna_cat.optimize(objective, n_trials=50)

# 최적 결과 출력
best_trial = optuna_cat.best_trial
print(f"Best AUC: {best_trial.value}")
print("Best hyperparameters:")
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")

#### GBC : Gradient Boosting Classifier

In [ ]:
sampler = TPESampler(seed=0)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),                          # 트리의 개수
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),                     # 학습률
        'max_depth': trial.suggest_int('max_depth', 3, 10),                                   # 트리의 최대 깊이
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),                   # 분할을 위한 최소 샘플 수
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),                     # 리프 노드에 최소한 있어야 할 샘플 수
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),                              # 샘플링 비율
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),    # 분할에 사용할 특성의 수
        'loss': trial.suggest_categorical('loss', ['log_loss', 'exponential']),               # 손실 함수
        'ccp_alpha': trial.suggest_float('ccp_alpha', 0.0, 0.1),                              # 가지치기 파라미터
        'random_state': 0,                                                                    # 고정된 랜덤 시드
        'validation_fraction': trial.suggest_float('validation_fraction', 0.1, 0.3),          # 검증 데이터 비율
        'n_iter_no_change': trial.suggest_int('n_iter_no_change', 5, 20),                     # 성능 향상이 없을 경우 훈련 중단
        'tol': trial.suggest_float('tol', 1e-4, 1e-2),                                        # 수렴을 위한 tolerance 값
        'min_impurity_decrease': trial.suggest_float('min_impurity_decrease', 0.0, 0.1),      # 불순도 감소 기준
        'max_leaf_nodes': trial.suggest_int('max_leaf_nodes', 10, 100),                       # 최대 리프 노드 수
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    auc_list = []

    for _, (train_index, valid_index) in enumerate(cv.split(train_feature,train_target)):
        X_train, y_train = train_feature.iloc[train_index], train_target.iloc[train_index]
        X_valid, y_valid = train_feature.iloc[valid_index], train_target.iloc[valid_index]


        gbc_model = GradientBoostingClassifier(**params)
        gbc_model.fit(X_train, y_train)

        # 예측 확률 계산
        y_prob = gbc_model.predict_proba(X_valid)[:,1]
        auc_score = roc_auc_score(y_valid, y_prob)
        auc_list.append(auc_score)

    # 평균 auc 반환
    mean_auc = np.mean(auc_list)
    return mean_auc

# optuna 최적화 실행(AUC 최대화)
optuna_gbc = optuna.create_study(direction='maximize', sampler=sampler)
optuna_gbc.optimize(objective, n_trials=50)

# 최적 결과 출력
best_trial = optuna_gbc.best_trial
print(f"Best AUC: {best_trial.value}")
print("Best hyperparameters:")
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")

#### LightGBM : Light Gradient Boosting Machine

In [ ]:
sampler = TPESampler(seed=0)

def objective(trial):
    params = {
        'num_boost_round': trial.suggest_int('num_boost_round', 100, 1000),                                   # 최대 부스팅 반복 횟수 (트리의 수) / default: 100
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),                                     # 학습률 (각 예제마다 가중치를 업데이트하는 비율) / default: 0.1
        'num_leaves': trial.suggest_int('num_leaves', 31, 128),                                               # 트리의 최대 잎 수 / default: 64
        'max_depth': trial.suggest_int('max_depth', -1, 15),                                                  # 트리의 최대 깊이 / default: 6
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 100),                                   # 각 리프 노드에 최소 데이터 수 / default: 3
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),                                # 각 트리에서 선택할 특징의 비율 / default: 0.9
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),                                # 배깅의 비율 (데이터의 일부를 무작위로 선택) / default: 0.9
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),                                             # 배깅을 수행하는 빈도 / default: 1
        'min_gain_to_split': trial.suggest_float('min_gain_to_split', 0.0, 1.0),                              # 각 트리에서 사용할 최소 게인 / default: 0.0
        'lambda_l1': trial.suggest_float('lambda_l1', 0.0, 0.1),                                              # L1 정규화 / default: 0.0
        'lambda_l2': trial.suggest_float('lambda_l2', 0.0, 0.1),                                              # L2 정규화 / default: 0.0
        'tree_learner': trial.suggest_categorical('tree_learner', ['serial', 'feature', 'data', 'voting']),   # 트리 훈련자 유형 / default: 'serial'
        'max_bin': trial.suggest_int('max_bin', 255, 512),                                                    # 최대 빈 수 (기능 값을 버킷화하는 데 사  용) / default: 255
        'early_stopping_rounds': trial.suggest_int('early_stopping_rounds', 10, 50),                          # 조기 중단 라운드 / default: 10
        'metric': 'auc',                                                                                      # 평가 지표 (auc 고정)
        'num_threads': trial.suggest_int('num_threads', 1, 8),                                                # 트리 학습에 사용하는 스레드 수 / default: 0
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 5.0),                                # 스케일링된 레이블의 가중치 (바이너리 분류에만 사용) / default: 1.0
        'verbosity': 0                                                                                        # 인쇄메시지
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    auc_list = []

    for _, (train_index, valid_index) in enumerate(cv.split(train_feature,train_target)):
        X_train, y_train = train_feature.iloc[train_index], train_target.iloc[train_index]
        X_valid, y_valid = train_feature.iloc[valid_index], train_target.iloc[valid_index]

        # LightGBM 모델 학습
        train_data = lgb.Dataset(X_train, label=y_train)
        valid_data = lgb.Dataset(X_valid, label=y_valid)

        # lightGBM 모델 학습
        lgb_model = lgb.train(
            params,
            train_data,
            valid_sets = [train_data, valid_data]
        )

        # 예측 확률 계산
        y_prob = lgb_model.predict(X_valid)
        auc_score = roc_auc_score(y_valid, y_prob)
        auc_list.append(auc_score)

    # 평균 auc 반환
    mean_auc = np.mean(auc_list)
    return mean_auc

# optuna 최적화 실행(AUC 최대화)
optuna_lgb = optuna.create_study(direction='maximize', sampler=sampler)
optuna_lgb.optimize(objective, n_trials=50)

# 최적 결과 출력
best_trial = optuna_lgb.best_trial
print(f"Best AUC: {best_trial.value}")
print("Best hyperparameters:")
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")

#### XGBoost : Extreme Gradient Boosting

In [ ]:
sampler = TPESampler(seed=0)

def objective(trial):
    params = {
        'num_round': trial.suggest_int('num_round', 100, 1000),                               # 부스팅 반복 횟수 (트리 개수) / default: 100
        'alpha': trial.suggest_float('alpha', 0.0, 1.0),                                      # L1 정규화 항 (Lasso 규제, 가중치 감소 효과) / default: 0.0
        'base_score': trial.suggest_float('base_score', 0.0, 1.0),                            # 초기 예측값 (모든 샘플의 기본 예측값) / default: 0.5
        'booster': trial.suggest_categorical('booster', ['gbtree']),                          # 부스팅 방식 ('gbtree': 트리 기반, 'gblinear': 선형 모델) / default: 'gbtree'
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.1, 1.0),              # 트리의 각 레벨에서 선택할 피처 비율 / default: 1.0
        'colsample_bynode': trial.suggest_float('colsample_bynode', 0.1, 1.0),                # 트리 각 노드에서 선택할 피처 비율 / default: 1.0
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.1, 1.0),                # 각 트리에서 선택할 피처 비율 (컬럼 샘플링) / default: 1.0
        'eta': trial.suggest_float('eta', 0.01, 0.3),                                         # 학습률 (트리 가중치 업데이트 비율) / default: 0.3
        'eval_metric': 'auc',                                                                 # 평가 지표 (AUC로 고정)
        'gamma': trial.suggest_float('gamma', 0.0, 1.0),                                      # 트리 분할을 위한 최소 손실 감소값 (클수록 보수적 분할) / default: 0.0
        'grow_policy': trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide']),  # 트리 성장 방식 ('depthwise': 깊이 우선, 'lossguide': 손실 기반) / default: 'depthwise'
        'lambda': trial.suggest_float('lambda', 0.0, 10.0),                                   # L2 정규화 항 (Ridge 규제, 과적합 방지) / default: 1.0
        'max_bin': trial.suggest_int('max_bin', 10, 512),                                     # 연속형 변수를 이산화할 때 사용할 빈 개수 / default: 256
        'max_delta_step': trial.suggest_int('max_delta_step', 0, 10),                         # 클래스 불균형 보정 (가중치 변화 제한) / default: 0
        'max_depth': trial.suggest_int('max_depth', 3, 12),                                   # 개별 트리의 최대 깊이 (깊을수록 모델 복잡도 증가) / default: 6
        'max_leaves': trial.suggest_int('max_leaves', 0, 50),                                 # 트리의 최대 리프 노드 개수 (0이면 제한 없음) / default: 0
        'min_child_weight': trial.suggest_float('min_child_weight', 0.1, 10.0),               # 리프 노드가 분할되기 위한 최소 가중치 합 (클수록 덜 분할) / default: 1.0
        'objective': 'binary:logistic',                                                       # 학습 목표 (이진 분류: 로지스틱 회귀) / default: 'binary:logistic'
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 0.1, 10.0),               # 불균형 데이터 조정 가중치 / default: 1.0
        'seed': trial.suggest_int('seed', 0, 1000),                                           # 난수 시드 (재현 가능성 확보) / default: 0
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),                              # 트리 학습 시 사용하는 샘플링 비율 (과적합 방지) / default: 1.0
        'verbosity': trial.suggest_int('verbosity', 0, 3),                                    # 출력 메시지 수준 (0: 없음, 3: 상세 로그) / default: 1
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    auc_list = []

    for _, (train_index, valid_index) in enumerate(cv.split(train_feature,train_target)):
        X_train, y_train = train_feature.iloc[train_index], train_target.iloc[train_index]
        X_valid, y_valid = train_feature.iloc[valid_index], train_target.iloc[valid_index]

        dtrain = xgb.DMatrix(X_train, label=y_train,enable_categorical=True)
        dvalid = xgb.DMatrix(X_valid, label=y_valid,enable_categorical=True)

        xgb_model = xgb.train(
            params,
            dtrain,
            evals=[(dtrain, 'train'), (dvalid, 'valid')],
            early_stopping_rounds=trial.suggest_int('early_stopping_rounds', 10, 100)
        )

        # 예측 확률 계산
        y_prob = xgb_model.predict(dvalid)
        auc_score = roc_auc_score(y_valid, y_prob)
        auc_list.append(auc_score)

    # 평균 auc 반환
    mean_auc = np.mean(auc_list)
    return mean_auc

# optuna 최적화 실행(AUC 최대화)
optuna_xgb = optuna.create_study(direction='maximize', sampler=sampler)
optuna_xgb.optimize(objective, n_trials=50)

# 최적 결과 출력
best_trial = optuna_xgb.best_trial
print(f"Best AUC: {best_trial.value}")
print("Best hyperparameters:")
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")

#### Ada : Ada Boost Classifier

In [ ]:
sampler = TPESampler(seed=0)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 10, 500),                         # 부스팅할 약한 학습기의 개수 / default: 50
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 1.0, log=True),         # 학습률 (작을수록 보수적 학습) / default: 1.0
        'algorithm': trial.suggest_categorical('algorithm', ['SAMME', 'SAMME.R']),          # 부스팅 알고리즘 ('SAMME': 분류기 확률 사용 안 함, 'SAMME.R': 확률 사용) / default: 'SAMME.R'
        'random_state': trial.suggest_int('random_state', 0, 1000)                          # 난수 시드 (재현 가능성 확보) / default: None
    }

    # 기본 학습기 (약한 학습기)로 결정 트리 사용 시 세부 하이퍼파라미터 설정
    base_estimator_params = {
        'max_depth': trial.suggest_int('max_depth', 1, 10),                                 # 트리의 최대 깊이 (깊을수록 복잡한 모델) / default: None (무제한)
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),                 # 노드를 분할하기 위한 최소 샘플 개수 / default: 2
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),                   # 리프 노드에 있어야 하는 최소 샘플 개수 / default: 1
        'max_features': trial.suggest_categorical('max_features', [None, 'sqrt', 'log2']),  # 트리 분할 시 사용할 최대 특성 개수 / default: None (모든 특성 사용)
        'max_leaf_nodes': trial.suggest_int('max_leaf_nodes', 10, 100, log=True),           # 리프 노드 최대 개수 (과적합 방지) / default: None (제한 없음)
        'min_impurity_decrease': trial.suggest_float('min_impurity_decrease', 0.0, 0.1)     # 최소 불순도 감소 기준 (값이 클수록 덜 분할) / default: 0.0
    }


    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    auc_list = []

    for _, (train_index, valid_index) in enumerate(cv.split(train_feature,train_target)):
        X_train, y_train = train_feature.iloc[train_index], train_target.iloc[train_index]
        X_valid, y_valid = train_feature.iloc[valid_index], train_target.iloc[valid_index]


        base_estimator = DecisionTreeClassifier(**base_estimator_params)


        ada_model = AdaBoostClassifier(estimator=  base_estimator, **params)
        ada_model.fit(X_train,y_train)

        # 예측 확률 계산
        y_prob = ada_model.predict_proba(X_valid)[:,1]
        auc_score = roc_auc_score(y_valid, y_prob)
        auc_list.append(auc_score)

    # 평균 auc 반환
    mean_auc = np.mean(auc_list)
    return mean_auc

# optuna 최적화 실행(AUC 최대화)
optuna_ada = optuna.create_study(direction='maximize', sampler=sampler)
optuna_ada.optimize(objective, n_trials=50)

# 최적 결과 출력
best_trial = optuna_ada.best_trial
print(f"Best AUC: {best_trial.value}")
print("Best hyperparameters:")
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")

### 보팅

- 하드 보팅
    - 예측값들 중 다수 분류기가 결정한 예측값을 최종 보팅 결괏값(다수결)
- **소프트 보팅**
    - **레이블 값 결정 확률을 모두 더하고 이를 평균해서 이들 중 확률이 가장 높은 레이블 값**
        - catboost: 0.8929926718960395
        - lightgbm: 0.8932907005145936 (가장 높은 성능)
        - gbc: 0.8872064342320748
        - xgboost: 0.8862153263809549
        - ada: 0.8885549835886202


#### 조합 생성

In [ ]:
from itertools import combinations

models = ['lightgbm','catboost','ada','gbc','xgboost']

model_combinations = []
for i in range(1,len(models) + 1) :
  model_combinations.extend(combinations(models,i))


for combo in model_combinations :
  print(combo)

#### 가중치

In [ ]:
roc_score = {
    'lightgbm' : 0.8932907005145936,
    'catboost' :  0.8929926718960395,
    'gbc' : 0.8872064342320748,
    'ada' : 0.8885549835886202,
    'xgboost' : 0.8862153263809549
}

total_roc = sum(roc_score.values())
weights = {model : roc_score[model] / total_roc for model in roc_score}


#### 소프트 보팅 구현

```
런타임 시간 초과로 인해 vscode로 작동

<출력 내용>
==================================================
All Voting Model Results:
==================================================
Combination: ('lightgbm', 'catboost', 'ada'), Mean ROC: 0.8938, Fold AUCs: [0.8934978365688402, 0.8938326442565854, 0.8955069275443579, 0.894669072207884, 0.8917147260538993], Weights: [0.20081800009368792, 0.20075100117484296, 0.1997533777915236]
Combination: ('lightgbm', 'catboost'), Mean ROC: 0.8937, Fold AUCs: [0.8933396310709194, 0.893783476366994, 0.8954482242974858, 0.8945306782152891, 0.8916242494785669], Weights: [0.20081800009368792, 0.20075100117484296]
Combination: ('lightgbm', 'ada'), Mean ROC: 0.8936, Fold AUCs: [0.8931520381850946, 0.8937203218194638, 0.8951289438908986, 0.894567478748641, 0.8915290573145307], Weights: [0.20081800009368792, 0.1997533777915236]
Combination: ('lightgbm', 'catboost', 'gbc'), Mean ROC: 0.8936, Fold AUCs: [0.8931513283979688, 0.893681525625472, 0.8950011574482155, 0.8945294788557031, 0.8916050514750801], Weights: [0.20081800009368792, 0.20075100117484296, 0.19945021445997535]
Combination: ('lightgbm', 'catboost', 'ada', 'gbc'), Mean ROC: 0.8936, Fold AUCs: [0.8931691391028265, 0.8935629691665392, 0.8949837704147432, 0.8945340122148723, 0.8915550288182447], Weights: [0.20081800009368792, 0.20075100117484296, 0.1997533777915236, 0.19945021445997535]
Combination: ('lightgbm',), Mean ROC: 0.8935, Fold AUCs: [0.8929317070558651, 0.8936245665841729, 0.8951016033309859, 0.8944121726855536, 0.8914025672875441], Weights: [0.20081800009368792]
Combination: ('lightgbm', 'catboost', 'ada', 'gbc', 'xgboost'), Mean ROC: 0.8932, Fold AUCs: [0.8926437645762213, 0.8934806558688347, 0.894813894695938, 0.8939221517699328, 0.8912955118688451], Weights: [0.20081800009368792, 0.20075100117484296, 0.1997533777915236, 0.19945021445997535, 0.1992274064799702]
Combination: ('lightgbm', 'catboost', 'gbc', 'xgboost'), Mean ROC: 0.8932, Fold AUCs: [0.8925730884860468, 0.8934165164228894, 0.8948244314583114, 0.8938301476857283, 0.89124866411829], Weights: [0.20081800009368792, 0.20075100117484296, 0.19945021445997535, 0.1992274064799702]
Combination: ('lightgbm', 'catboost', 'ada', 'xgboost'), Mean ROC: 0.8930, Fold AUCs: [0.8924189546352047, 0.8932285438834858, 0.8947979987656527, 0.8935150242069827, 0.8910287427071797], Weights: [0.20081800009368792, 0.20075100117484296, 0.1997533777915236, 0.1992274064799702]
Combination: ('catboost', 'ada'), Mean ROC: 0.8929, Fold AUCs: [0.8928716337547731, 0.8928225896652617, 0.8946143509757141, 0.8935299199229416, 0.8908564283829623], Weights: [0.20075100117484296, 0.1997533777915236]
Combination: ('lightgbm', 'gbc'), Mean ROC: 0.8929, Fold AUCs: [0.8923737208370478, 0.8930987793906342, 0.8941116098548648, 0.8939569166929784, 0.8909556318237375], Weights: [0.20081800009368792, 0.19945021445997535]
Combination: ('lightgbm', 'ada', 'gbc'), Mean ROC: 0.8928, Fold AUCs: [0.892402513984567, 0.8929673092077901, 0.893997933870201, 0.8939302226897158, 0.8908832142473857], Weights: [0.20081800009368792, 0.1997533777915236, 0.19945021445997535]
Combination: ('lightgbm', 'catboost', 'xgboost'), Mean ROC: 0.8928, Fold AUCs: [0.8921338953218712, 0.8930538977349274, 0.8946034675731166, 0.8932709270229854, 0.890813349801908], Weights: [0.20081800009368792, 0.20075100117484296, 0.1992274064799702]
Combination: ('lightgbm', 'ada', 'gbc', 'xgboost'), Mean ROC: 0.8927, Fold AUCs: [0.8920538819545438, 0.8929867540737049, 0.8941728386233714, 0.8933690269348105, 0.8907872627470123], Weights: [0.20081800009368792, 0.1997533777915236, 0.19945021445997535, 0.1992274064799702]
Combination: ('catboost',), Mean ROC: 0.8926, Fold AUCs: [0.8923956362023391, 0.8926753941211314, 0.8943304251208917, 0.8931307972599805, 0.89062105062568], Weights: [0.20075100117484296]
Combination: ('lightgbm', 'gbc', 'xgboost'), Mean ROC: 0.8926, Fold AUCs: [0.89181337416337, 0.8929566458942237, 0.8941201548115049, 0.8931950867849445, 0.8906811042255813], Weights: [0.20081800009368792, 0.19945021445997535, 0.1992274064799702]
Combination: ('catboost', 'ada', 'gbc', 'xgboost'), Mean ROC: 0.8923, Fold AUCs: [0.891702757416241, 0.892531819041566, 0.8938949019413129, 0.8927961841879624, 0.890540500447106], Weights: [0.20075100117484296, 0.1997533777915236, 0.19945021445997535, 0.1992274064799702]
Combination: ('lightgbm', 'ada', 'xgboost'), Mean ROC: 0.8922, Fold AUCs: [0.8914786077423184, 0.8925623508924323, 0.8940337918756247, 0.8927116073305519, 0.8902973532484585], Weights: [0.20081800009368792, 0.1997533777915236, 0.1992274064799702]
Combination: ('catboost', 'gbc'), Mean ROC: 0.8921, Fold AUCs: [0.891932310275882, 0.8919342718193735, 0.8935210697172062, 0.8930547589625584, 0.8901473017852636], Weights: [0.20075100117484296, 0.19945021445997535]
Combination: ('catboost', 'ada', 'gbc'), Mean ROC: 0.8921, Fold AUCs: [0.891860154086972, 0.8920371799481813, 0.8933044085725734, 0.8929669416336058, 0.8901454859809784], Weights: [0.20075100117484296, 0.1997533777915236, 0.19945021445997535]
Combination: ('catboost', 'gbc', 'xgboost'), Mean ROC: 0.8920, Fold AUCs: [0.8913817575640969, 0.8924157826020412, 0.8935931213638265, 0.8924840260982847, 0.890266435053677], Weights: [0.20075100117484296, 0.19945021445997535, 0.1992274064799702]
Combination: ('lightgbm', 'xgboost'), Mean ROC: 0.8918, Fold AUCs: [0.8909726395660479, 0.8922141755471491, 0.8936885987367154, 0.8922751009593941, 0.8898934798583946], Weights: [0.20081800009368792, 0.1992274064799702]
Combination: ('catboost', 'ada', 'xgboost'), Mean ROC: 0.8914, Fold AUCs: [0.8907609634379737, 0.8917075416015589, 0.89329524186442, 0.8916315675184154, 0.8896598243639644], Weights: [0.20075100117484296, 0.1997533777915236, 0.1992274064799702]
Combination: ('ada', 'gbc', 'xgboost'), Mean ROC: 0.8909, Fold AUCs: [0.8901326092536258, 0.8912470383169282, 0.8924324901106295, 0.8912600851424269, 0.889187814293712], Weights: [0.1997533777915236, 0.19945021445997535, 0.1992274064799702]
Combination: ('catboost', 'xgboost'), Mean ROC: 0.8906, Fold AUCs: [0.8898434342754069, 0.8910431643450134, 0.8925744007668959, 0.8908151832541674, 0.8889602334899811], Weights: [0.20075100117484296, 0.1992274064799702]
Combination: ('gbc', 'xgboost'), Mean ROC: 0.8903, Fold AUCs: [0.889455422815459, 0.8907662208147088, 0.8917602639290056, 0.8906178555923745, 0.8886633549918068], Weights: [0.19945021445997535, 0.1992274064799702]
Combination: ('ada',), Mean ROC: 0.8898, Fold AUCs: [0.8899362623265814, 0.8899483562188512, 0.8901956097388339, 0.8908433241912423, 0.8880789851533679], Weights: [0.1997533777915236]
Combination: ('ada', 'xgboost'), Mean ROC: 0.8882, Fold AUCs: [0.8872365347219718, 0.8886614433817226, 0.8901844402204958, 0.8881892844703235, 0.8866266883788437], Weights: [0.1997533777915236, 0.1992274064799702]
Combination: ('ada', 'gbc'), Mean ROC: 0.8881, Fold AUCs: [0.8878776100523287, 0.8882713190670742, 0.8886591682113615, 0.8894122873949454, 0.8864398586252201], Weights: [0.1997533777915236, 0.19945021445997535]
Combination: ('gbc',), Mean ROC: 0.8872, Fold AUCs: [0.8869048798096002, 0.8871286608343953, 0.8877862620998898, 0.8885442371404584, 0.8855423010646644], Weights: [0.19945021445997535]
Combination: ('xgboost',), Mean ROC: 0.8859, Fold AUCs: [0.8845785112376798, 0.8865306156607695, 0.8880430014571215, 0.8857380850639736, 0.8844179990708364], Weights: [0.1992274064799702]
==================================================
Best Model Combination: ('lightgbm', 'catboost', 'ada')
Best AUC: 0.8938
Fold AUCs: [0.8934978365688402, 0.8938326442565854, 0.8955069275443579, 0.894669072207884, 0.8917147260538993]
Best Weights: [0.20081800009368792, 0.20075100117484296, 0.1997533777915236]
==================================================
```

In [ ]:
models_dict = {
    'lightgbm': LGBMClassifier(
        n_estimators=633,
        learning_rate=0.03342462805497592,
        num_leaves=56,
        max_depth=14,
        min_data_in_leaf=52,
        feature_fraction=0.6364882192462488,
        bagging_fraction=0.8584003437311537,
        bagging_freq=2,
        min_gain_to_split=0.14446805858185238,
        reg_alpha=0.07945719422821337,
        reg_lambda=0.09228860158082569,
        tree_learner='feature',
        max_bin=277,
        num_threads=8,
        scale_pos_weight=4.955235602871113
    ),
    'gbc': GradientBoostingClassifier(
        n_estimators=473,
        learning_rate=0.021945351322294467,
        max_depth=10,
        min_samples_split=2,
        min_samples_leaf=3,
        subsample=0.9750612643600708,
        max_features='sqrt',
        loss='exponential',
        ccp_alpha=0.00015702247402449507,
        validation_fraction=0.2984753709846782,
        n_iter_no_change=15,
        tol=0.005547267099418254,
        min_impurity_decrease=0.09411712873217182,
        max_leaf_nodes=86
    ),
    'ada': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=5,
            min_samples_split=8,
            min_samples_leaf=3,
            max_features=None,
            max_leaf_nodes=12,
            min_impurity_decrease=0.00011591970091207504,
            random_state=582
        ),
        n_estimators=339,
        learning_rate=0.028444996574740207,
        algorithm='SAMME.R',
        random_state=582
    ),
    'xgboost': XGBClassifier(
        n_estimators=360,
        alpha=0.7558611471242505,
        base_score=0.5566055700158926,
        booster='gbtree',
        colsample_bylevel=0.652980718854562,
        colsample_bynode=0.6207456906259616,
        colsample_bytree=0.7084669537044522,
        learning_rate=0.2931701770417505,
        eval_metric='auc',
        gamma=0.38038567264515777,
        grow_policy='lossguide',
        reg_lambda=7.6102845156108465,
        max_bin=142,
        max_delta_step=7,
        max_depth=11,
        max_leaves=33,
        min_child_weight=3.6279534873415384,
        objective='binary:logistic',
        scale_pos_weight=4.199769049496863,
        seed=675,
        subsample=0.7974634701882836,
        verbosity=1,
    ),
    'catboost': CatBoostClassifier(
        iterations=567,
        learning_rate=0.02900858676615259,
        depth=7,
        l2_leaf_reg=1.2647515608981543,
        random_strength=0.23498389010132387,
        bagging_temperature=0.024432250592590334,
        grow_policy='Depthwise',
        border_count=174,
        od_wait=50,
        verbose=0,
        random_state=42
    )
}

# 소프트 보팅 성능 평가
best_combo = None
best_auc = 0
results = []

# 교차검증 설정
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for combo in model_combinations:
    estimators = [(name, models_dict[name]) for name in combo]
    weights_for_combo = [weights[name] for name in combo]

    voting_clf = VotingClassifier(
        estimators=estimators,
        voting='soft',
        weights=weights_for_combo
    )

    roc_list = []  # 교차검증 roc 기록

    # 교차 검증 루프
    for train_idx, valid_idx in cv.split(train_feature, train_target):
        X_train, y_train = train_feature.iloc[train_idx], train_target.iloc[train_idx]
        X_valid, y_valid = train_feature.iloc[valid_idx], train_target.iloc[valid_idx]

        # 모델 학습
        voting_clf.fit(X_train, y_train)

        # 예측 및 ROC 계산
        y_prob = voting_clf.predict_proba(X_valid)[:, 1]
        roc = roc_auc_score(y_valid, y_prob)
        roc_list.append(roc)

    # 교차 검증 평균 AUC 계산
    mean_roc = sum(roc_list) / len(roc_list)

    # 결과 저장
    results.append({
        "combination": combo,
        "mean_roc": mean_roc,
        "roc_per_fold": roc_list,  # 폴드별 점수 추가
        "weights": weights_for_combo
    })

# 결과 정렬 (ROC 기준 내림차순)
results = sorted(results, key=lambda x: x["mean_roc"], reverse=True)

# 모든 결과 출력
print("=" * 50)
print("All Voting Model Results:")
print("=" * 50)
for result in results:
    print(f"Combination: {result['combination']}, Mean ROC: {result['mean_roc']:.4f}, "
          f"Fold AUCs: {result['roc_per_fold']}, Weights: {result['weights']}")

# 최적의 결과 출력
best_result = results[0]  # 가장 높은 점수
print("=" * 50)
print(f"Best Model Combination: {best_result['combination']}")
print(f"Best AUC: {best_result['mean_roc']:.4f}")
print(f"Fold AUCs: {best_result['roc_per_fold']}")
print(f"Best Weights: {best_result['weights']}")
print("=" * 50)

## 3.최종 모델
```
Best Model Combination: ('lightgbm', 'catboost', 'ada')
Best AUC: 0.8938
Fold AUCs: [0.8934978365688402, 0.8938326442565854, 0.8955069275443579, 0.894669072207884, 0.8917147260538993]
Best Weights: [0.20081800009368792, 0.20075100117484296, 0.1997533777915236]
```

### 데이터 분리

In [ ]:
# 데이터 분리
X_train, X_valid, y_train, y_valid = train_test_split(train_feature, train_target,
                                                      test_size = 0.3,
                                                      random_state=0)

### 모델 생성

```
<최종점수>
Final Model AUC Score: 0.8941
```

In [ ]:
final_models = {
    'lightgbm': LGBMClassifier(
        n_estimators=633,
        learning_rate=0.03342462805497592,
        num_leaves=56,
        max_depth=14,
        min_data_in_leaf=52,
        feature_fraction=0.6364882192462488,
        bagging_fraction=0.8584003437311537,
        bagging_freq=2,
        min_gain_to_split=0.14446805858185238,
        reg_alpha=0.07945719422821337,
        reg_lambda=0.09228860158082569,
        tree_learner='feature',
        max_bin=277,
        num_threads=8,
        scale_pos_weight=4.955235602871113
    ),
    'catboost': CatBoostClassifier(
        iterations=567,
        learning_rate=0.02900858676615259,
        depth=7,
        l2_leaf_reg=1.2647515608981543,
        random_strength=0.23498389010132387,
        bagging_temperature=0.024432250592590334,
        grow_policy='Depthwise',
        border_count=174,
        od_wait=50,
        verbose=0,
        random_state=42
    ),
    'ada': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=5,
            min_samples_split=8,
            min_samples_leaf=3,
            max_features=None,
            max_leaf_nodes=12,
            min_impurity_decrease=0.00011591970091207504,
            random_state=582
        ),
        n_estimators=339,
        learning_rate=0.028444996574740207,
        algorithm='SAMME.R',
        random_state=582
    )
}

# 소프트 보팅 모델 생성
final_voting_model = VotingClassifier(
    estimators=[('lightgbm', final_models['lightgbm']), ('catboost', final_models['catboost']),('ada',final_models['ada'])],
    voting='soft',
    weights=[0.20081800009368792, 0.20075100117484296, 0.1997533777915236]  # 최적 가중치 적용
)
# 학습
final_voting_model.fit(X_train, y_train)

# 예측
y_prob = final_voting_model.predict_proba(X_valid)[:, 1]
auc_score = roc_auc_score(y_valid, y_prob)

# 결과 출력
print(f"Final Model AUC Score: {auc_score:.4f}")

### 제출 파일 생성

In [ ]:
# 테스트 데이터 예측
test_prob = final_voting_model.predict_proba(test)[:, 1]

submission['Exited'] = test_prob
submission.to_csv('fourth_noedit_submission.csv',index=False)

### 변수 중요도

In [ ]:
# 1. Feature Importance 계산 및 시각화(각 모델 feature importance 평균 사용)

def plot_feature_importance(model, model_columns):
    feature_importance = np.zeros(len(model_columns))
    for name, estimator in final_voting_model.named_estimators_.items():                            # named_estimators_ : 개별 추정기(estimator)들을 딕셔너리 형태로 저장
        if hasattr(estimator, "feature_importances_"):                                              # hasattr(객체, "속성_이름") : 객체가 속성을 갖고 있으면 True, 아니면 False 반환
            feature_importance += estimator.feature_importances_                                    # 각각의 모델 변수 중요도 더하기
    feature_importance /= len(final_voting_model.named_estimators_)                                 # 모델 개수로 나누기

    # Feature 중요도 시각화
    sorted_idx = np.argsort(feature_importance)[::-1]                                               # np.argsort() : 정렬된 값의 인덱스 반환 [start:stop:step] / [:: -1] -> 내림차순으로 변경
    sorted_features = np.array(model_columns)[sorted_idx]                                           # 변수 중요도에 따라 변수 명 정렬(인덱스따라)
    sorted_importances = feature_importance[sorted_idx]                                             # 변수 중요도에 따라 값 정렬

    plt.figure(figsize=(10, 8))
    bars = plt.barh(range(len(model_columns)), sorted_importances[::-1], align="center")            # barh : 가로 막대그래프 생성(기본적으로 y값 작은것부터 위로 쌓임) / sorted_importances[::-1] : 막대 길이 조정 / align : 막대 중심 위치
    plt.yticks(range(len(model_columns)), sorted_features[::-1])                                    # sorted_features[::-1] 사용 이유 : 위와 동일. 기본적으로 작은것부터 위로 쌓임
    plt.xlabel("Feature Importance")
    plt.title("Feature Importance from Model")

    # 막대에 값 표시
    for bar, value in zip(bars, sorted_importances[::-1]):
        plt.text(bar.get_width(), bar.get_y() + bar.get_height()/2, f'{value:.2f}', va='center')    # plt.text(x, y, s, va='center') / bar.get_width() : 텍스트 x좌표, bar.get_y() + bar.get_height()/2 : 텍스트 y좌표, f'{value:.2f}', va='center' : 표시 할 텍스트

    plt.show()

# 2. Permutation Importance 계산 및 시각화
def plot_permutation_importance(model, X_valid, y_valid, feature_names):
    perm_importance = permutation_importance(
        model, X_valid, y_valid, scoring='roc_auc', n_repeats=5, random_state=42
    )

    # Feature 중요도 정렬
    sorted_idx = np.argsort(perm_importance.importances_mean)[::-1]                                 # 중요도 높은 순서로 정렬
    sorted_features = np.array(feature_names)[sorted_idx]                                           # 정렬된 변수명
    sorted_importances = perm_importance.importances_mean[sorted_idx]                               # 정렬된 변수 중요도

    # Feature 중요도 시각화
    plt.figure(figsize=(10, 8))
    bars = plt.barh(range(len(feature_names)), sorted_importances[::-1], align="center")            # 내림차순 정렬한 막대그래프
    plt.yticks(range(len(feature_names)), sorted_features[::-1])                                    # y축 변수명도 동일한 순서로 정렬
    plt.xlabel("Permutation Importance")
    plt.title("Permutation Importance from Validation Set")

    # 막대에 값 표시
    for bar, value in zip(bars, sorted_importances[::-1]):
        plt.text(bar.get_width(), bar.get_y() + bar.get_height()/2, f'{value:.4f}', va='center')

    plt.show()

# Feature Importance 그래프 생성
plot_feature_importance(final_voting_model, train_feature.columns)

# Permutation Importance 그래프 생성
plot_permutation_importance(final_voting_model, X_valid, y_valid, train_feature.columns)

#### feature importance
- **Top 5**
    - EstimatedSalary_Scaled
    - CreditScore_Scaled
    - Surname_tfidf_0
    - Balance_Scaled
    - Surname_tfidf_1

In [ ]:
from IPython.display import Image
Image('/content/drive/MyDrive/파이널 프로젝트_이정수/분류/SVD_feature_importance.png')

#### permutation importance
- **Top 5**
    - NumOfProducts_2
    - Age_Scaled
    - Balance_Scaled
    - IsActiveMember
    - Geography_Germany

In [ ]:
from IPython.display import Image
Image('/content/drive/MyDrive/파이널 프로젝트_이정수/분류/SVD_permutation_importance.png')

#### 중요도 해석

- **결론**
    - Balance_Scaled가 공통적으로 포함
    - **제거 할 변수**
        - **Products_Per_Tenure**
            - Feature importance : 568.14
            - Permutation importance : 0.0001
            - VIF : 12.44187

# --

## 4.중요도 기반 재모델링
- 제거
    - Products_Per_Tenure

In [ ]:
# 1. 전처리

train = pd.read_csv('/content/drive/MyDrive/파이널 프로젝트_이정수/분류/train.csv')
test = pd.read_csv('/content/drive/MyDrive/파이널 프로젝트_이정수/분류/test.csv')
submission = pd.read_csv('/content/drive/MyDrive/파이널 프로젝트_이정수/분류/sample_submission.csv')

# 스케일링
numeric_cols = ['Age','CreditScore','Balance','EstimatedSalary']

# 로그 변환과 스케일링을 포함한 기능 정의
scaled_features = {
    'CreditScore': RobustScaler(),
    'EstimatedSalary': StandardScaler(),
    'Age': ('log', RobustScaler()),
    'Balance': ('log1p', MinMaxScaler())
}

# 스케일링 적용
for feature, scaled_info in scaled_features.items() :
  if isinstance(scaled_info, tuple) :
    log_info, scaler = scaled_info
    if log_info == 'log' :
      train_feature = np.log(train[[feature]])
      test_feature = np.log(test[[feature]])
    elif log_info == 'log1p' :
      train_feature = np.log1p(train[[feature]])
      test_feature = np.log1p(test[[feature]])
  else :
    scaler = scaled_info
    train_feature = train[[feature]]
    test_feature = test[[feature]]


  # 스케일링 적용
  scaled_feature = feature + '_Scaled'
  train[scaled_feature] = scaler.fit_transform(train_feature)
  test[scaled_feature] = scaler.transform(test_feature)

# 파생변수
# 변수 합치기
train['Sur_Geo_Gend_Sal'] = train['CustomerId'].astype('str') + train['Surname'] + train['Geography'] + train['Gender'] + np.round(train.EstimatedSalary).astype('str')
test['Sur_Geo_Gend_Sal'] = test['CustomerId'].astype('str') + test['Surname'] + test['Geography'] + test['Gender'] + np.round(test.EstimatedSalary).astype('str')

# 특정 텍스트 열(col_name)을 TF-IDF로 벡터화하고, 차원축소를 통해 주요 특징만 추출하는 과정
def get_vectors(train,test,col_name):

    # TF-IDF 벡터화
    '''
    TF-IDF 코드 분해
    : 상위 1000개 주요 단어만 벡터화 사용
    : 훈련데이터에 대해 벡터화 학습 및 변환 수행(벡터화 -> 텍스트를 중요도 점수로 표시) ->  # TF-IDF 점수 계산
    : 테스트 데이터 훈련데이터 기준으로 변환 수행
    '''
    vectorizer = TfidfVectorizer(max_features=1000)
    vectors_train = vectorizer.fit_transform(train[col_name])
    vectors_test = vectorizer.transform(test[col_name])

    # TruncatedSVD 차원축소
    '''
    TruncatedSVD 코드 설명 및 분해
    <TruncatedSVD 설명>
    : 차원 축소, U⋅Σ⋅V^T -> 이 중 중요한 k개의 차원만 남긴다
    : 학습 -> 데이터의 주요 패턴(특이값) 학습 -> 학습된 U,Σ,𝑉^T를 사용해 주요 정보만 남기기
    : 변환 -> 학습 정보 바탕으로 원래 vectors_train을 저차원 벡터 공간으로 변환

    ```
    원래 TF-IDF 행렬 : 책을 단어들의 숫자로 표현
    - 행 : 문서 /  열 : 단어 /  값 : TF-IDF 점수
    - 크기 (1000,5000)

    SVD 학습 및 변환 후 : "이 책은 과학, 우주 기술"과 같이 책의 핵심 주제를 3개의 숫자로 요약
    - 행 : 문서 / 열 : 중요 특성(3차원)
    - 크기 (1000,3)
    ```
    <TruncatedSVD 코드 분해>
    : TF-IDF 행렬을 3개의 중요 성분으로 압축 -> 데이터 크기 줄이기, 모델 학습속도 개선, 과적합방지
    : vectors_train에 대해 SVD 학습(fit) 및 변환
    '''
    svd = TruncatedSVD(3)
    x_sv_train = svd.fit_transform(vectors_train)
    x_sv_test = svd.transform(vectors_test) # 테스트 데이터 반환


    # 데이터 프레임 변환
    tfidf_df_train = pd.DataFrame(x_sv_train)
    tfidf_df_test = pd.DataFrame(x_sv_test)


    # 열이름 지정
    cols = [(col_name + "_tfidf_" + str(f)) for f in tfidf_df_train.columns.to_list()]
    tfidf_df_train.columns = cols
    tfidf_df_test.columns = cols

    # Reset the index of the DataFrames before concatenation
    train = train.reset_index(drop=True)
    test = test.reset_index(drop=True)

    # Concatenate transformed features with original data
    train = pd.concat([train, tfidf_df_train], axis="columns")
    test = pd.concat([test, tfidf_df_test], axis="columns")
    return train,test

train,test = get_vectors(train,test,'Surname')
train,test = get_vectors(train,test,'Sur_Geo_Gend_Sal')

# 파생변수 생성
def feature_data(df):
    # senior 생성(60대 이상은 1, 아니면 0)
    df['Senior'] = df['Age'].apply(lambda x: 1 if x >= 60 else 0)
    # 신용카드 소유자와 활성화 고객 여부
    df['Active_by_CreditCard'] = df['HasCrCard'] * df['IsActiveMember']
    # 새로운 연령대 범주(20살 단위)
    df['AgeCat'] = np.round(df.Age/20).astype('int').astype('category')

    cat_cols = ['Geography', 'Gender', 'NumOfProducts','AgeCat']
    #onehotEncoding
    df=pd.get_dummies(df,columns=cat_cols)
    return df

train = feature_data(train)
test = feature_data(test)

feat_cols = train.columns.drop(['id','CustomerId','Surname','Exited','Sur_Geo_Gend_Sal'])
feat_cols = feat_cols.drop(numeric_cols)

feat_cols_test = test.columns.drop(['id','CustomerId','Surname','Sur_Geo_Gend_Sal'])
feat_cols_test = feat_cols_test.drop(numeric_cols)

train_feature = train[feat_cols]
train_target = train['Exited']
test = test[feat_cols_test]

for i in train_feature.columns :
  if train_feature[i].dtype == 'bool' :
    train_feature[i] = train_feature[i].astype('int')

for i in test.columns :
  if test[i].dtype == 'bool' :
    test[i] = test[i].astype('int')

In [ ]:
# 2. 모델링

# 데이터 분리
X_train, X_valid, y_train, y_valid = train_test_split(train_feature, train_target,
                                                      test_size = 0.3,
                                                      random_state=0)

In [ ]:
# 2. 모델링
# AutoML

# 인덱스 번호 새로 할당
X_train_automl = X_train.reset_index(drop=True)
y_train_automl = y_train.reset_index(drop=True)

# X_train, y_train 합치기
Xy_train_automl = pd.concat([X_train_automl, y_train_automl], axis=1)

# AutoML 모델 세팅
clf = setup(data=Xy_train_automl,
            target = 'Exited',
            train_size = 0.7,
            data_split_shuffle=True,
            session_id = 0,
            fold = 5
            )

# AutoML top5 model 설정
top5_model = compare_models(fold=5, round=3, n_select=5, sort = 'AUC', errors='ignore',verbose=True)
top5_model

###optuna
- CatBoost
```
Best AUC: 0.8930844927272021
Best hyperparameters:
  iterations: 535
  learning_rate: 0.06964529408502032
  depth: 6
  l2_leaf_reg: 0.027235876696900984
  random_strength: 4.836212370043395
  bagging_temperature: 1.9799984680862954
  grow_policy: Lossguide
  border_count: 166
  od_wait: 28
```
- LightGBM
```
Best AUC: 0.8927221334546112
Best hyperparameters:
  num_boost_round: 500
  learning_rate: 0.029888796349948284
  num_leaves: 65
  max_depth: 9
  min_data_in_leaf: 32
  feature_fraction: 0.6358713698824439
  bagging_fraction: 0.8661769682816917
  bagging_freq: 9
  min_gain_to_split: 0.045680108972356276
  lambda_l1: 0.04081755440857283
  lambda_l2: 0.026788484813059954
  tree_learner: data
  max_bin: 268
  early_stopping_rounds: 23
  num_threads: 4
  scale_pos_weight: 3.4872207884352115
```
- GBC
```
Best AUC: 0.8873325360161234
Best hyperparameters:
  n_estimators: 473
  learning_rate: 0.021945351322294467
  max_depth: 10
  min_samples_split: 2
  min_samples_leaf: 3
  subsample: 0.9750612643600708
  max_features: sqrt
  loss: exponential
  ccp_alpha: 0.00015702247402449507
  validation_fraction: 0.2984753709846782
  n_iter_no_change: 15
  tol: 0.005547267099418254
  min_impurity_decrease: 0.09411712873217182
  max_leaf_nodes: 86
```
- XGBoost
```
Best AUC: 0.8882229217305595
Best hyperparameters:
  num_round: 106
  alpha: 0.6330437539243525
  base_score: 0.39870903675669067
  booster: gbtree
  colsample_bylevel: 0.9923227855716938
  colsample_bynode: 0.4151277034733893
  colsample_bytree: 0.997498141356141
  eta: 0.27192150582147717
  eval_metric: auc
  gamma: 0.04712099219003624
  grow_policy: lossguide
  lambda: 2.852528179903329
  max_bin: 341
  max_delta_step: 4
  max_depth: 7
  max_leaves: 0
  min_child_weight: 4.542860426764745
  objective: binary:logistic
  scale_pos_weight: 1.6658131783101051
  seed: 141
  subsample: 0.6819570352186918
  verbosity: 3
  early_stopping_rounds: 24
```
- AdaBoost
```
Best AUC: 0.8886618978806397
Best hyperparameters:
  n_estimators: 266
  learning_rate: 0.09909004448349125
  algorithm: SAMME.R
  random_state: 519
  max_depth: 5
  min_samples_split: 9
  min_samples_leaf: 6
  max_features: sqrt
  max_leaf_nodes: 41
  min_impurity_decrease: 0.00015191581774246914
```

#### GBC : Gradient Boosting Classifier

In [ ]:
sampler = TPESampler(seed=0)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'loss': trial.suggest_categorical('loss', ['log_loss', 'exponential']),
        'ccp_alpha': trial.suggest_float('ccp_alpha', 0.0, 0.1),
        'random_state': 0,
        'validation_fraction': trial.suggest_float('validation_fraction', 0.1, 0.3),
        'n_iter_no_change': trial.suggest_int('n_iter_no_change', 5, 20),
        'tol': trial.suggest_float('tol', 1e-4, 1e-2),
        'min_impurity_decrease': trial.suggest_float('min_impurity_decrease', 0.0, 0.1),
        'max_leaf_nodes': trial.suggest_int('max_leaf_nodes', 10, 100),
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    auc_list = []

    for _, (train_index, valid_index) in enumerate(cv.split(train_feature,train_target)):
        X_train, y_train = train_feature.iloc[train_index], train_target.iloc[train_index]
        X_valid, y_valid = train_feature.iloc[valid_index], train_target.iloc[valid_index]


        gbc_model = GradientBoostingClassifier(**params)
        gbc_model.fit(X_train, y_train)

        # 예측 확률 계산
        y_prob = gbc_model.predict_proba(X_valid)[:,1]
        auc_score = roc_auc_score(y_valid, y_prob)
        auc_list.append(auc_score)

    # 평균 auc 반환
    mean_auc = np.mean(auc_list)
    return mean_auc

# optuna 최적화 실행(AUC 최대화)
optuna_gbc = optuna.create_study(direction='maximize', sampler=sampler)
optuna_gbc.optimize(objective, n_trials=50)

# 최적 결과 출력
best_trial = optuna_gbc.best_trial
print(f"Best AUC: {best_trial.value}")
print("Best hyperparameters:")
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")

#### lightGBM : Light Gradient Boosting Machine

In [ ]:
sampler = TPESampler(seed=0)

def objective(trial):
    params = {
        'num_boost_round': trial.suggest_int('num_boost_round', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 31, 128),
        'max_depth': trial.suggest_int('max_depth', -1, 15),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 100),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
        'min_gain_to_split': trial.suggest_float('min_gain_to_split', 0.0, 1.0),
        'lambda_l1': trial.suggest_float('lambda_l1', 0.0, 0.1),
        'lambda_l2': trial.suggest_float('lambda_l2', 0.0, 0.1),
        'tree_learner': trial.suggest_categorical('tree_learner', ['serial', 'feature', 'data', 'voting']),
        'max_bin': trial.suggest_int('max_bin', 255, 512),
        'early_stopping_rounds': trial.suggest_int('early_stopping_rounds', 10, 50),
        'metric': 'auc',
        'num_threads': trial.suggest_int('num_threads', 1, 8),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 5.0),
        'verbosity': 0
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    auc_list = []

    for _, (train_index, valid_index) in enumerate(cv.split(train_feature,train_target)):
        X_train, y_train = train_feature.iloc[train_index], train_target.iloc[train_index]
        X_valid, y_valid = train_feature.iloc[valid_index], train_target.iloc[valid_index]

        # LightGBM 모델 학습
        train_data = lgb.Dataset(X_train, label=y_train)
        valid_data = lgb.Dataset(X_valid, label=y_valid)

        # lightGBM 모델 학습
        lgb_model = lgb.train(
            params,
            train_data,
            valid_sets = [train_data, valid_data]
        )

        # 예측 확률 계산
        y_prob = lgb_model.predict(X_valid)
        auc_score = roc_auc_score(y_valid, y_prob)
        auc_list.append(auc_score)

    # 평균 auc 반환
    mean_auc = np.mean(auc_list)
    return mean_auc

# optuna 최적화 실행(AUC 최대화)
optuna_lgb = optuna.create_study(direction='maximize', sampler=sampler)
optuna_lgb.optimize(objective, n_trials=50)

# 최적 결과 출력
best_trial = optuna_lgb.best_trial
print(f"Best AUC: {best_trial.value}")
print("Best hyperparameters:")
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")

#### XGBoost : Extreme Gradient Boosting

In [ ]:
sampler = TPESampler(seed=0)

def objective(trial):
    params = {
        'num_round': trial.suggest_int('num_round', 100, 1000),
        'alpha': trial.suggest_float('alpha', 0.0, 1.0),
        'base_score': trial.suggest_float('base_score', 0.0, 1.0),
        'booster': trial.suggest_categorical('booster', ['gbtree']),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.1, 1.0),
        'colsample_bynode': trial.suggest_float('colsample_bynode', 0.1, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.1, 1.0),
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'eval_metric': trial.suggest_categorical('eval_metric', ['auc']),
        'gamma': trial.suggest_float('gamma', 0.0, 1.0),
        'grow_policy': trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide']),
        'lambda': trial.suggest_float('lambda', 0.0, 10.0),
        'max_bin': trial.suggest_int('max_bin', 10, 512),
        'max_delta_step': trial.suggest_int('max_delta_step', 0, 10),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'max_leaves': trial.suggest_int('max_leaves', 0, 50),
        'min_child_weight': trial.suggest_float('min_child_weight', 0.1, 10.0),
        'objective': trial.suggest_categorical('objective', ['binary:logistic']),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 0.1, 10.0),
        'seed': trial.suggest_int('seed', 0, 1000),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'verbosity': trial.suggest_int('verbosity', 0, 3),
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    auc_list = []

    for _, (train_index, valid_index) in enumerate(cv.split(train_feature,train_target)):
        X_train, y_train = train_feature.iloc[train_index], train_target.iloc[train_index]
        X_valid, y_valid = train_feature.iloc[valid_index], train_target.iloc[valid_index]

        dtrain = xgb.DMatrix(X_train, label=y_train,enable_categorical=True)
        dvalid = xgb.DMatrix(X_valid, label=y_valid,enable_categorical=True)

        xgb_model = xgb.train(
            params,
            dtrain,
            evals=[(dtrain, 'train'), (dvalid, 'valid')],
            early_stopping_rounds=trial.suggest_int('early_stopping_rounds', 10, 100)
        )

        # 예측 확률 계산
        y_prob = xgb_model.predict(dvalid)
        auc_score = roc_auc_score(y_valid, y_prob)
        auc_list.append(auc_score)

    # 평균 auc 반환
    mean_auc = np.mean(auc_list)
    return mean_auc

# optuna 최적화 실행(AUC 최대화)
optuna_xgb = optuna.create_study(direction='maximize', sampler=sampler)
optuna_xgb.optimize(objective, n_trials=50)

# 최적 결과 출력
best_trial = optuna_xgb.best_trial
print(f"Best AUC: {best_trial.value}")
print("Best hyperparameters:")
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")

#### Ada : Ada Boost Classifier

In [ ]:
sampler = TPESampler(seed=0)

def objective(trial):
    params = {
        # AdaBoost의 핵심 하이퍼파라미터들
        'n_estimators': trial.suggest_int('n_estimators', 10, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 1.0, log=True),
        'algorithm': trial.suggest_categorical('algorithm', ['SAMME', 'SAMME.R']),
        'random_state': trial.suggest_int('random_state', 0, 1000)
    }

    # 기본 학습기 (약한 학습기)로 결정 트리 사용 시 세부 파라미터 설정
    base_estimator_params = {
        'max_depth': trial.suggest_int('max_depth', 1, 10),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features': trial.suggest_categorical('max_features', [None, 'sqrt', 'log2']),
        'max_leaf_nodes': trial.suggest_int('max_leaf_nodes', 10, 100, log=True),
        'min_impurity_decrease': trial.suggest_float('min_impurity_decrease', 0.0, 0.1)
    }


    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    auc_list = []

    for _, (train_index, valid_index) in enumerate(cv.split(train_feature,train_target)):
        X_train, y_train = train_feature.iloc[train_index], train_target.iloc[train_index]
        X_valid, y_valid = train_feature.iloc[valid_index], train_target.iloc[valid_index]


        base_estimator = DecisionTreeClassifier(**base_estimator_params)


        ada_model = AdaBoostClassifier(estimator=  base_estimator, **params)
        ada_model.fit(X_train,y_train)

        # 예측 확률 계산
        y_prob = ada_model.predict_proba(X_valid)[:,1]
        auc_score = roc_auc_score(y_valid, y_prob)
        auc_list.append(auc_score)

    # 평균 auc 반환
    mean_auc = np.mean(auc_list)
    return mean_auc

# optuna 최적화 실행(AUC 최대화)
optuna_ada = optuna.create_study(direction='maximize', sampler=sampler)
optuna_ada.optimize(objective, n_trials=50)

# 최적 결과 출력
best_trial = optuna_ada.best_trial
print(f"Best AUC: {best_trial.value}")
print("Best hyperparameters:")
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")

####Catboost : CatBoost Classifier

In [ ]:
sampler = TPESampler(seed=0)

def objective(trial):
    # 하이퍼파라미터 정의
    params = {
        "iterations": trial.suggest_int("iterations", 100, 1000),
        "learning_rate": trial.suggest_loguniform("learning_rate", 1e-4, 1.0),
        "depth": trial.suggest_int("depth", 1, 16),
        "l2_leaf_reg": trial.suggest_loguniform("l2_leaf_reg", 1e-2, 10.0),
        "random_strength": trial.suggest_loguniform("random_strength", 1e-2, 10.0),
        "bagging_temperature": trial.suggest_loguniform("bagging_temperature", 1e-5, 10.0),
        "grow_policy": trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]),
        "border_count": trial.suggest_int("border_count", 1, 255),
        "thread_count": -1,
        "random_seed": 42,
        "eval_metric": "AUC",
        "verbose": 0,
        "od_type": "Iter",
        "od_wait": trial.suggest_int("od_wait", 10, 50),
        "task_type": "CPU",
        "loss_function": "Logloss"
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    auc_list = []

    # 범주형 컬럼 자동 감지
    cat_features_indices = [
        train_feature.columns.get_loc(col)
        for col in train_feature.select_dtypes(include=['category']).columns
    ]

    for _, (train_index, valid_index) in enumerate(cv.split(train_feature, train_target)):
        X_train, y_train = train_feature.iloc[train_index], train_target.iloc[train_index]
        X_valid, y_valid = train_feature.iloc[valid_index], train_target.iloc[valid_index]

        # CatBoostClassifier 초기화 및 학습
        cat_model = CatBoostClassifier(**params)
        cat_model.fit(
            X_train, y_train,
            eval_set=(X_valid, y_valid),
            early_stopping_rounds=50,
            verbose=0,
            cat_features=cat_features_indices  # 범주형 컬럼 지정
        )

        # 예측 확률 계산
        y_score = cat_model.predict_proba(X_valid)[:, 1]
        auc_score = roc_auc_score(y_valid, y_score)
        auc_list.append(auc_score)

    # 평균 auc 반환
    mean_auc = np.mean(auc_list)
    return mean_auc

# optuna 최적화 실행(AUC 최대화)
optuna_cat = optuna.create_study(direction='maximize', sampler=sampler)
optuna_cat.optimize(objective, n_trials=50)

# 최적 결과 출력
best_trial = optuna_cat.best_trial
print(f"Best AUC: {best_trial.value}")
print("Best hyperparameters:")
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")


### 보팅

#### 조합 생성

In [ ]:
models = ['lightgbm','gbc','ada','xgboost','catboost']

model_combinations = []
for i in range(1,len(models) + 1) :
  model_combinations.extend(combinations(models,i))


# for combo in model_combinations :
#   print(combo)

####가중치

In [ ]:
roc_score = {
    'lightgbm' : 0.8927221334546112,
    'gbc' : 0.8873325360161234,
    'ada' : 0.8886618978806397,
    'xgboost' : 0.8882229217305595,
    'catboost' :  0.8930844927272021
}

total_roc = sum(roc_score.values())
weights = {model : roc_score[model] / total_roc for model in roc_score}

#### 소프트 보팅 구현

In [ ]:
models_dict = {
    'lightgbm': LGBMClassifier(
        n_estimators=500,
        learning_rate=0.029888796349948284,
        num_leaves=65,
        max_depth=9,
        min_data_in_leaf=32,
        feature_fraction=0.6358713698824439,
        bagging_fraction=0.8661769682816917,
        bagging_freq=9,
        min_split_gain=0.045680108972356276,
        reg_alpha=0.04081755440857283,
        reg_lambda=0.026788484813059954,
        tree_learner='data',
        max_bin=268,
        num_threads=4,
        scale_pos_weight=3.4872207884352115
    ),
    'gbc': GradientBoostingClassifier(
        n_estimators=473,
        learning_rate=0.021945351322294467,
        max_depth=10,
        min_samples_split=2,
        min_samples_leaf=3,
        subsample=0.9750612643600708,
        max_features='sqrt',
        loss='exponential',
        ccp_alpha=0.00015702247402449507,
        validation_fraction=0.2984753709846782,
        n_iter_no_change=15,
        tol=0.005547267099418254,
        min_impurity_decrease=0.09411712873217182,
        max_leaf_nodes=86
    ),
    'ada': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=5,
            min_samples_split=9,
            min_samples_leaf=6,
            max_features='sqrt',
            max_leaf_nodes=41,
            min_impurity_decrease=0.00015191581774246914
        ),
        n_estimators=266,
        learning_rate=0.09909004448349125,
        algorithm='SAMME.R',
        random_state=519
    ),
    'xgboost': XGBClassifier(
        n_estimators=106,
        alpha=0.6330437539243525,
        base_score=0.39870903675669067,
        booster='gbtree',
        colsample_bylevel=0.9923227855716938,
        colsample_bynode=0.4151277034733893,
        colsample_bytree=0.997498141356141,
        learning_rate=0.27192150582147717,
        eval_metric='auc',
        gamma=0.04712099219003624,
        grow_policy='lossguide',
        reg_lambda=2.852528179903329,
        max_bin=341,
        max_delta_step=4,
        max_depth=7,
        max_leaves=0,
        min_child_weight=4.542860426764745,
        objective='binary:logistic',
        scale_pos_weight=1.6658131783101051,
        seed=141,
        subsample=0.6819570352186918,
        verbosity=3
    ),
    'catboost': CatBoostClassifier(
        iterations=535,
        learning_rate=0.06964529408502032,
        depth=6,
        l2_leaf_reg=0.027235876696900984,
        random_strength=4.836212370043395,
        bagging_temperature=1.9799984680862954,
        grow_policy='Lossguide',
        border_count=166,
        od_wait=28,
        verbose=0,
        random_state=42
    )
}

# 소프트 보팅 성능 평가
best_combo = None
best_auc = 0
results = []

# 교차검증 설정
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for combo in model_combinations:
    estimators = [(name, models_dict[name]) for name in combo]
    weights_for_combo = [weights[name] for name in combo]

    voting_clf = VotingClassifier(
        estimators=estimators,
        voting='soft',
        weights=weights_for_combo
    )

    roc_list = []

    # 교차 검증 루프
    for train_idx, valid_idx in cv.split(train_feature, train_target):
        X_train, y_train = train_feature.iloc[train_idx], train_target.iloc[train_idx]
        X_valid, y_valid = train_feature.iloc[valid_idx], train_target.iloc[valid_idx]

        # 모델 학습
        voting_clf.fit(X_train, y_train)

        # 예측 및 ROC 계산
        y_prob = voting_clf.predict_proba(X_valid)[:, 1]
        roc = roc_auc_score(y_valid, y_prob)
        roc_list.append(roc)

    # 교차 검증 평균 AUC 계산
    mean_roc = sum(roc_list) / len(roc_list)

    # 결과 저장
    results.append({
        "combination": combo,
        "mean_roc": mean_roc,
        "roc_per_fold": roc_list,
        "weights": weights_for_combo
    })

# 결과 정렬 (ROC 기준 내림차순)
results = sorted(results, key=lambda x: x["mean_roc"], reverse=True)

# 모든 결과 출력
print("=" * 50)
print("All Voting Model Results:")
print("=" * 50)
for result in results:
    print(f"Combination: {result['combination']}, Mean ROC: {result['mean_roc']:.4f}, "
          f"Fold AUCs: {result['roc_per_fold']}, Weights: {result['weights']}")

# 최적의 결과 출력
best_result = results[0]
print("=" * 50)
print(f"Best Model Combination: {best_result['combination']}")
print(f"Best AUC: {best_result['mean_roc']:.4f}")
print(f"Fold AUCs: {best_result['roc_per_fold']}")
print(f"Best Weights: {best_result['weights']}")
print("=" * 50)

### 최종모델
```
Best Model Combination: ('lightgbm', 'ada', 'catboost')
Best AUC: 0.8942
Fold AUCs: [0.89347933808576, 0.8939094168129269, 0.8958242243985506, 0.8955682948083079, 0.8919839217903821]
Best Weights: [0.20061063425812803, 0.19969822668671516, 0.2006920628693158]
```

- 모델 생성
```
Final Model AUC Score: 0.8937
```

In [ ]:
# 데이터 분리
X_train, X_valid, y_train, y_valid = train_test_split(train_feature, train_target,
                                                      test_size = 0.3,
                                                      random_state=0)

final_models = {
    'lightgbm': LGBMClassifier(
        n_estimators=500,
        learning_rate=0.029888796349948284,
        num_leaves=65,
        max_depth=9,
        min_data_in_leaf=32,
        feature_fraction=0.6358713698824439,
        bagging_fraction=0.8661769682816917,
        bagging_freq=9,
        min_split_gain=0.045680108972356276,
        reg_alpha=0.04081755440857283,
        reg_lambda=0.026788484813059954,
        tree_learner='data',
        max_bin=268,
        num_threads=4,
        scale_pos_weight=3.4872207884352115
    ),
    'ada': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=5,
            min_samples_split=9,
            min_samples_leaf=6,
            max_features='sqrt',
            max_leaf_nodes=41,
            min_impurity_decrease=0.00015191581774246914
        ),
        n_estimators=266,
        learning_rate=0.09909004448349125,
        algorithm='SAMME.R',
        random_state=519
    ),
    'catboost': CatBoostClassifier(
        iterations=535,
        learning_rate=0.06964529408502032,
        depth=6,
        l2_leaf_reg=0.027235876696900984,
        random_strength=4.836212370043395,
        bagging_temperature=1.9799984680862954,
        grow_policy='Lossguide',
        border_count=166,
        od_wait=28,
        verbose=0,
        random_state=42
    )
}

# 소프트 보팅 모델 생성
final_voting_model = VotingClassifier(
    estimators=[('lightgbm', final_models['lightgbm']), ('ada', final_models['ada']),('catboost',final_models['catboost'])],
    voting='soft',
    weights=[0.20061063425812803, 0.19969822668671516, 0.2006920628693158]  # 최적 가중치 적용
)

# 학습
final_voting_model.fit(X_train, y_train)

# 예측
y_prob = final_voting_model.predict_proba(X_valid)[:, 1]
auc_score = roc_auc_score(y_valid, y_prob)

# 결과 출력
print(f"Final Model AUC Score: {auc_score:.4f}")

#### 중요도 기반 재모델링 제출 파일 생성

In [ ]:
# 테스트 데이터 예측
test_prob = final_voting_model.predict_proba(test)[:, 1]

submission['Exited'] = test_prob
submission.to_csv('svd_edit_submission.csv',index=False)

# 최종모델_1
- Final Model AUC Score: 0.8936

## 패키지

In [ ]:
# 기본 패키지
import pandas as pd  # 데이터프레임 생성 및 조작
import numpy as np  # 수치 계산 및 배열 처리

# 데이터 시각화
import matplotlib.pyplot as plt  # 2D 그래프, 플롯
import seaborn as sns  # 통계적 그래프 작성
import plotly.express as px  # 대화형 그래프 (라인플롯, 히스토그램, 파이차트, 지리데이터 등)
from plotly.subplots import make_subplots  # 그래프 세밀하게 조정 및 커스터마이징
import plotly.graph_objects as go  # 대화형 그래프 객체

# 그래프를 Jupyter Notebook에 출력
%matplotlib inline

# 데이터 전처리
from sklearn.preprocessing import LabelEncoder  # 레이블 인코딩
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler  # 데이터 스케일링
# pd.get_dummies로 OneHotEncoder 대체 가능

# 데이터 분할
from sklearn.model_selection import train_test_split  # 데이터 분리
from sklearn.model_selection import StratifiedKFold  # 층화 교차 검증

# 필요한 패키지 import(분산팽창계수(VIF))
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 텍스트 데이터 처리
from sklearn.feature_extraction.text import TfidfVectorizer  # 텍스트 데이터 TF-IDF 벡터화
from sklearn.decomposition import TruncatedSVD  # 차원 축소

# 머신러닝 모델들
from sklearn.ensemble import GradientBoostingClassifier  # Gradient Boosting 모델
from sklearn.ensemble import RandomForestClassifier  # Random Forest 모델
from sklearn.ensemble import AdaBoostClassifier  # AdaBoost 모델
from sklearn.tree import DecisionTreeClassifier  # 의사결정 트리 모델

# LightGBM
from lightgbm import LGBMClassifier  # LightGBM 분류기
import lightgbm as lgb  # LightGBM 라이브러리

# CatBoost
from catboost import CatBoostClassifier, Pool  # CatBoost 분류기 및 데이터 준비 객체
from catboost.utils import eval_metric  # CatBoost 평가 지표 계산

# 성능 평가 지표
from sklearn.metrics import roc_auc_score  # ROC AUC 점수 계산

# 모델 앙상블
from sklearn.ensemble import VotingClassifier  # 앙상블 학습을 위한 VotingClassifier

# 기타 도구
from itertools import combinations  # 조합 생성
from sklearn.inspection import permutation_importance  # 변수 중요도 계산 (Permutation Importance)

# 경고 무시
import warnings  # 경고 메시지 제어
warnings.filterwarnings('ignore')

# 출력 관련
from IPython.display import Image  # Jupyter Notebook에서 이미지 표시

# PyCaret: AutoML (자동화된 머신러닝 워크플로우)
from pycaret.classification import *

# 설정
pd.set_option('display.max_columns', None)  # 최대 컬럼 설정

In [ ]:
# 데이터 불러오기
train = pd.read_csv('/content/drive/MyDrive/파이널 프로젝트_이정수/분류/train.csv')
test = pd.read_csv('/content/drive/MyDrive/파이널 프로젝트_이정수/분류/test.csv')
submission = pd.read_csv('/content/drive/MyDrive/파이널 프로젝트_이정수/분류/sample_submission.csv')

# 스케일링
numeric_cols = ['Age','CreditScore','Balance','EstimatedSalary']

# 로그 변환과 스케일링을 포함한 기능 정의
scaled_features = {
    'CreditScore': RobustScaler(),
    'EstimatedSalary': StandardScaler(),
    'Age': ('log', RobustScaler()),        # 로그 변환 후 스케일링
    'Balance': ('log1p', MinMaxScaler())   # 로그 변환 후 스케일링
}

# 스케일링 적용
for feature, scaled_info in scaled_features.items() :
  if isinstance(scaled_info, tuple) :
    log_info, scaler = scaled_info
    if log_info == 'log' :
      train_feature = np.log(train[[feature]])
      test_feature = np.log(test[[feature]])
    elif log_info == 'log1p' :
      train_feature = np.log1p(train[[feature]])
      test_feature = np.log1p(test[[feature]])
  else :
    scaler = scaled_info
    train_feature = train[[feature]]
    test_feature = test[[feature]]


  # 스케일링 적용
  scaled_feature = feature + '_Scaled'
  train[scaled_feature] = scaler.fit_transform(train_feature)
  test[scaled_feature] = scaler.transform(test_feature)

# 파생변수
# 변수 합치기
train['Sur_Geo_Gend_Sal'] = train['CustomerId'].astype('str') + train['Surname'] + train['Geography'] + train['Gender'] + np.round(train.EstimatedSalary).astype('str')
test['Sur_Geo_Gend_Sal'] = test['CustomerId'].astype('str') + test['Surname'] + test['Geography'] + test['Gender'] + np.round(test.EstimatedSalary).astype('str')

# tf-idf, svd
# 특정 텍스트 열(col_name)을 TF-IDF로 벡터화하고, 차원축소를 통해 주요 특징만 추출하는 과정
def get_vectors(train,test,col_name):

    # TF-IDF 벡터화
    vectorizer = TfidfVectorizer(max_features=1000)
    vectors_train = vectorizer.fit_transform(train[col_name])
    vectors_test = vectorizer.transform(test[col_name])

    # TruncatedSVD 차원축소
    svd = TruncatedSVD(3)
    x_sv_train = svd.fit_transform(vectors_train)
    x_sv_test = svd.transform(vectors_test) # 테스트 데이터 반환


    # 데이터 프레임 변환
    tfidf_df_train = pd.DataFrame(x_sv_train)
    tfidf_df_test = pd.DataFrame(x_sv_test)


    # 열이름 지정
    cols = [(col_name + "_tfidf_" + str(f)) for f in tfidf_df_train.columns.to_list()]
    tfidf_df_train.columns = cols
    tfidf_df_test.columns = cols

    # Reset the index of the DataFrames before concatenation
    train = train.reset_index(drop=True)
    test = test.reset_index(drop=True)

    # Concatenate transformed features with original data
    train = pd.concat([train, tfidf_df_train], axis="columns")
    test = pd.concat([test, tfidf_df_test], axis="columns")
    return train,test
train,test = get_vectors(train,test,'Surname')
train,test = get_vectors(train,test,'Sur_Geo_Gend_Sal')

# 파생변수 생성
def feature_data(df):
    # senior 생성(60대 이상은 1, 아니면 0)
    df['Senior'] = df['Age'].apply(lambda x: 1 if x >= 60 else 0)
    # 신용카드 소유자와 활성화 고객 여부
    df['Active_by_CreditCard'] = df['HasCrCard'] * df['IsActiveMember']
    # 상품수 대비 이용기간
    df['Products_Per_Tenure'] =  df['Tenure'] / df['NumOfProducts']
    # 새로운 연령대 범주(20살 단위)
    df['AgeCat'] = np.round(df.Age/20).astype('int').astype('category')

    cat_cols = ['Geography', 'Gender', 'NumOfProducts','AgeCat']
    #onehotEncoding
    df=pd.get_dummies(df,columns=cat_cols)
    return df

train = feature_data(train)
test = feature_data(test)

feat_cols = train.columns.drop(['id','CustomerId','Surname','Exited','Sur_Geo_Gend_Sal'])
feat_cols = feat_cols.drop(numeric_cols)

feat_cols_test = test.columns.drop(['id','CustomerId','Surname','Sur_Geo_Gend_Sal'])
feat_cols_test = feat_cols_test.drop(numeric_cols)

train_feature = train[feat_cols]
train_target = train['Exited']
test = test[feat_cols_test]

for i in train_feature.columns :
  if train_feature[i].dtype == 'bool' :
    train_feature[i] = train_feature[i].astype('int')

for i in test.columns :
  if test[i].dtype == 'bool' :
    test[i] = test[i].astype('int')

# 데이터 분리
X_train, X_valid, y_train, y_valid = train_test_split(train_feature, train_target,
                                                      test_size = 0.3,
                                                      random_state=0)

final_models = {
    'lightgbm': LGBMClassifier(
        n_estimators=633,  # Optuna 결과
        learning_rate=0.03342462805497592,
        num_leaves=56,
        max_depth=14,
        min_data_in_leaf=52,
        feature_fraction=0.6364882192462488,
        bagging_fraction=0.8584003437311537,
        bagging_freq=2,
        min_gain_to_split=0.14446805858185238,
        reg_alpha=0.07945719422821337,
        reg_lambda=0.09228860158082569,
        tree_learner='feature',
        max_bin=277,
        num_threads=8,
        scale_pos_weight=4.955235602871113
    ),
    'catboost': CatBoostClassifier(
        iterations=567,  # Optuna 결과
        learning_rate=0.02900858676615259,
        depth=7,
        l2_leaf_reg=1.2647515608981543,
        random_strength=0.23498389010132387,
        bagging_temperature=0.024432250592590334,
        grow_policy='Depthwise',
        border_count=174,
        od_wait=50,
        verbose=0,  # 학습 출력 비활성화
        random_state=42
    ),
    'ada': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=5,
            min_samples_split=8,
            min_samples_leaf=3,
            max_features=None,
            max_leaf_nodes=12,
            min_impurity_decrease=0.00011591970091207504,
            random_state=582
        ),
        n_estimators=339,  # Optuna 결과
        learning_rate=0.028444996574740207,
        algorithm='SAMME.R',
        random_state=582
    )
}

# 소프트 보팅 모델 생성
final_voting_model_1 = VotingClassifier(
    estimators=[('lightgbm', final_models['lightgbm']), ('catboost', final_models['catboost']),('ada',final_models['ada'])],
    voting='soft',
    weights=[0.20081800009368792, 0.20075100117484296, 0.1997533777915236]  # 최적 가중치 적용
)

# 학습
final_voting_model_1.fit(X_train, y_train)

# 예측
y_prob = final_voting_model_1.predict_proba(X_valid)[:, 1]
auc_score = roc_auc_score(y_valid, y_prob)

# 결과 출력
print(f"Final Model AUC Score: {auc_score:.4f}")

# 테스트 데이터 예측
test_prob = final_voting_model_1.predict_proba(test)[:, 1]

submission['Exited'] = test_prob
submission.to_csv('fourth_noedit_submission.csv',index=False)

# 최종모델_2
- Final Model AUC Score: 0.8939


## 패키지

In [ ]:
# 기본 패키지
import pandas as pd  # 데이터프레임 생성 및 조작
import numpy as np  # 수치 계산 및 배열 처리

# 데이터 시각화
import matplotlib.pyplot as plt  # 2D 그래프, 플롯
import seaborn as sns  # 통계적 그래프 작성
import plotly.express as px  # 대화형 그래프 (라인플롯, 히스토그램, 파이차트, 지리데이터 등)
from plotly.subplots import make_subplots  # 그래프 세밀하게 조정 및 커스터마이징
import plotly.graph_objects as go  # 대화형 그래프 객체

# 그래프를 Jupyter Notebook에 출력
%matplotlib inline

# 데이터 전처리
from sklearn.preprocessing import LabelEncoder  # 레이블 인코딩
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler  # 데이터 스케일링
# pd.get_dummies로 OneHotEncoder 대체 가능

# 데이터 분할
from sklearn.model_selection import train_test_split  # 데이터 분리
from sklearn.model_selection import StratifiedKFold  # 층화 교차 검증

# 필요한 패키지 import(분산팽창계수(VIF))
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 텍스트 데이터 처리
from sklearn.feature_extraction.text import TfidfVectorizer  # 텍스트 데이터 TF-IDF 벡터화
from sklearn.decomposition import TruncatedSVD  # 차원 축소

# 머신러닝 모델들
from sklearn.ensemble import GradientBoostingClassifier  # Gradient Boosting 모델
from sklearn.ensemble import RandomForestClassifier  # Random Forest 모델
from sklearn.ensemble import AdaBoostClassifier  # AdaBoost 모델
from sklearn.tree import DecisionTreeClassifier  # 의사결정 트리 모델

# LightGBM
from lightgbm import LGBMClassifier  # LightGBM 분류기
import lightgbm as lgb  # LightGBM 라이브러리

# CatBoost
from catboost import CatBoostClassifier, Pool  # CatBoost 분류기 및 데이터 준비 객체
from catboost.utils import eval_metric  # CatBoost 평가 지표 계산

# 성능 평가 지표
from sklearn.metrics import roc_auc_score  # ROC AUC 점수 계산

# 모델 앙상블
from sklearn.ensemble import VotingClassifier  # 앙상블 학습을 위한 VotingClassifier

# 기타 도구
from itertools import combinations  # 조합 생성
from sklearn.inspection import permutation_importance  # 변수 중요도 계산 (Permutation Importance)

# 경고 무시
import warnings  # 경고 메시지 제어
warnings.filterwarnings('ignore')

# 출력 관련
from IPython.display import Image  # Jupyter Notebook에서 이미지 표시

# PyCaret: AutoML (자동화된 머신러닝 워크플로우)
from pycaret.classification import *

# 설정
pd.set_option('display.max_columns', None)  # 최대 컬럼 설정

In [ ]:
# 데이터 불러오기
train = pd.read_csv('/content/drive/MyDrive/파이널 프로젝트_이정수/분류/train.csv')
test = pd.read_csv('/content/drive/MyDrive/파이널 프로젝트_이정수/분류/test.csv')
submission = pd.read_csv('/content/drive/MyDrive/파이널 프로젝트_이정수/분류/sample_submission.csv')

# 스케일링
numeric_cols = ['Age','CreditScore','Balance','EstimatedSalary']

# 로그 변환과 스케일링을 포함한 기능 정의
scaled_features = {
    'CreditScore': RobustScaler(),
    'EstimatedSalary': StandardScaler(),
    'Age': ('log', RobustScaler()),        # 로그 변환 후 스케일링
    'Balance': ('log1p', MinMaxScaler())   # 로그 변환 후 스케일링
}

# 스케일링 적용
for feature, scaled_info in scaled_features.items() :
  if isinstance(scaled_info, tuple) :
    log_info, scaler = scaled_info
    if log_info == 'log' :
      train_feature = np.log(train[[feature]])
      test_feature = np.log(test[[feature]])
    elif log_info == 'log1p' :
      train_feature = np.log1p(train[[feature]])
      test_feature = np.log1p(test[[feature]])
  else :
    scaler = scaled_info
    train_feature = train[[feature]]
    test_feature = test[[feature]]


  # 스케일링 적용
  scaled_feature = feature + '_Scaled'
  train[scaled_feature] = scaler.fit_transform(train_feature)
  test[scaled_feature] = scaler.transform(test_feature)

# 파생변수
# 변수 합치기
train['Sur_Geo_Gend_Sal'] = train['CustomerId'].astype('str') + train['Surname'] + train['Geography'] + train['Gender'] + np.round(train.EstimatedSalary).astype('str')
test['Sur_Geo_Gend_Sal'] = test['CustomerId'].astype('str') + test['Surname'] + test['Geography'] + test['Gender'] + np.round(test.EstimatedSalary).astype('str')

# tf-idf, svd
# 특정 텍스트 열(col_name)을 TF-IDF로 벡터화하고, 차원축소를 통해 주요 특징만 추출하는 과정
def get_vectors(train,test,col_name):

    # TF-IDF 벡터화
    vectorizer = TfidfVectorizer(max_features=1000)
    vectors_train = vectorizer.fit_transform(train[col_name])
    vectors_test = vectorizer.transform(test[col_name])

    # TruncatedSVD 차원축소
    svd = TruncatedSVD(3)
    x_sv_train = svd.fit_transform(vectors_train)
    x_sv_test = svd.transform(vectors_test) # 테스트 데이터 반환


    # 데이터 프레임 변환
    tfidf_df_train = pd.DataFrame(x_sv_train)
    tfidf_df_test = pd.DataFrame(x_sv_test)


    # 열이름 지정
    cols = [(col_name + "_tfidf_" + str(f)) for f in tfidf_df_train.columns.to_list()]
    tfidf_df_train.columns = cols
    tfidf_df_test.columns = cols

    # Reset the index of the DataFrames before concatenation
    train = train.reset_index(drop=True)
    test = test.reset_index(drop=True)

    # Concatenate transformed features with original data
    train = pd.concat([train, tfidf_df_train], axis="columns")
    test = pd.concat([test, tfidf_df_test], axis="columns")
    return train,test
train,test = get_vectors(train,test,'Surname')
train,test = get_vectors(train,test,'Sur_Geo_Gend_Sal')

# 파생변수 생성
def feature_data(df):
    # senior 생성(60대 이상은 1, 아니면 0)
    df['Senior'] = df['Age'].apply(lambda x: 1 if x >= 60 else 0)
    # 신용카드 소유자와 활성화 고객 여부
    df['Active_by_CreditCard'] = df['HasCrCard'] * df['IsActiveMember']
    # 새로운 연령대 범주(20살 단위)
    df['AgeCat'] = np.round(df.Age/20).astype('int').astype('category')

    cat_cols = ['Geography', 'Gender', 'NumOfProducts','AgeCat']
    #onehotEncoding
    df=pd.get_dummies(df,columns=cat_cols)
    return df

train = feature_data(train)
test = feature_data(test)

feat_cols = train.columns.drop(['id','CustomerId','Surname','Exited','Sur_Geo_Gend_Sal'])
feat_cols = feat_cols.drop(numeric_cols)

feat_cols_test = test.columns.drop(['id','CustomerId','Surname','Sur_Geo_Gend_Sal'])
feat_cols_test = feat_cols_test.drop(numeric_cols)

train_feature = train[feat_cols]
train_target = train['Exited']
test = test[feat_cols_test]

for i in train_feature.columns :
  if train_feature[i].dtype == 'bool' :
    train_feature[i] = train_feature[i].astype('int')

for i in test.columns :
  if test[i].dtype == 'bool' :
    test[i] = test[i].astype('int')

# 데이터 분리
X_train, X_valid, y_train, y_valid = train_test_split(train_feature, train_target,
                                                      test_size = 0.3,
                                                      random_state=0)

final_models = {
    'lightgbm': LGBMClassifier(
        n_estimators=500,  # num_boost_round 대신 n_estimators로 변경
        learning_rate=0.029888796349948284,
        num_leaves=65,
        max_depth=9,
        min_data_in_leaf=32,
        feature_fraction=0.6358713698824439,
        bagging_fraction=0.8661769682816917,
        bagging_freq=9,
        min_split_gain=0.045680108972356276,  # min_gain_to_split
        reg_alpha=0.04081755440857283,  # lambda_l1
        reg_lambda=0.026788484813059954,  # lambda_l2
        tree_learner='data',
        max_bin=268,
        num_threads=4,
        scale_pos_weight=3.4872207884352115
    ),
    'ada': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=5,
            min_samples_split=9,
            min_samples_leaf=6,
            max_features='sqrt',
            max_leaf_nodes=41,
            min_impurity_decrease=0.00015191581774246914
        ),
        n_estimators=266,
        learning_rate=0.09909004448349125,
        algorithm='SAMME.R',
        random_state=519
    ),
    'catboost': CatBoostClassifier(
        iterations=535,
        learning_rate=0.06964529408502032,
        depth=6,
        l2_leaf_reg=0.027235876696900984,
        random_strength=4.836212370043395,
        bagging_temperature=1.9799984680862954,
        grow_policy='Lossguide',
        border_count=166,
        od_wait=28,
        verbose=0,  # 학습 출력 비활성화
        random_state=42
    )
}

# 소프트 보팅 모델 생성
final_voting_model_2 = VotingClassifier(
    estimators=[('lightgbm', final_models['lightgbm']), ('ada', final_models['ada']),('catboost',final_models['catboost'])],
    voting='soft',
    weights=[0.20061063425812803, 0.19969822668671516, 0.2006920628693158]  # 최적 가중치 적용
)

# 학습
final_voting_model_2.fit(X_train, y_train)

# 예측
y_prob = final_voting_model_2.predict_proba(X_valid)[:, 1]
auc_score = roc_auc_score(y_valid, y_prob)

# 결과 출력
print(f"Final Model AUC Score: {auc_score:.4f}")
# 테스트 데이터 예측
test_prob = final_voting_model_2.predict_proba(test)[:, 1]

submission['Exited'] = test_prob
submission.to_csv('fourth_edit_submission.csv',index=False)